In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import random
import gc
import sys
import pprint
import scrublet as scr

In [2]:
os.chdir("/home/lishaogang/NEU_2026")

In [ ]:

adata = sc.read_h5ad('/home/lishaogang/NEU_2026/final_merged_adata.h5ad')

In [7]:
adata.obs

In [3]:
adata_obs  = adata.obs.copy()

In [4]:
#对每个细胞计算指控指标
import scanpy as sc

# 1. 标记线粒体基因 (通常以 MT- 开头，注意大小写，小鼠通常是 mt-)
adata.var['mt'] = adata.var_names.str.startswith('MT-') 

# 2. 计算 QC 指标
# qc_vars=['mt'] 会自动计算名为 'pct_counts_mt' 的列
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

# 查看计算出的新列
print(adata.obs.columns)
# 你会发现多了：n_genes_by_counts, total_counts, pct_counts_mt 等

In [6]:
import scanpy as sc
import matplotlib.pyplot as plt

# 设置全局绘图参数，这里设置宽度和高度
sc.set_figure_params(figsize=(12, 5))

# 绘图（去掉 figsize 参数）
sc.pl.violin(adata, 
             keys=['n_genes_by_counts', 'pct_counts_mt'], 
             groupby='dataset', 
             jitter=0.4, 
             multi_panel=True, 
             rotation=90)

In [21]:
import scanpy as sc
import matplotlib.pyplot as plt

# 设置全局绘图参数，这里设置宽度和高度
sc.set_figure_params(figsize=(12, 5))

# 绘图（去掉 figsize 参数）
sc.pl.violin(adata, 
             keys=['n_genes_by_counts', 'pct_counts_mt'], 
             groupby='dataset', 
             jitter=0.4, 
             multi_panel=True, 
             rotation=90)

In [7]:
# 1. 过滤基因数过少的 (建议调高到 500)
sc.pp.filter_cells(adata, min_genes=400)


# 3. 过滤线粒体过高的 (20% 作为多数据集整合的底线)
adata = adata[adata.obs.pct_counts_mt < 20, :].copy()

# 4. 顺便过滤掉在少于 3 个细胞中表达的无效基因
sc.pp.filter_genes(adata, min_cells=3)

In [3]:
# This section demonstrates how to add a custom module path.
# The 'anno_self' module contains user-defined annotation functions.
# Ensure this path points to the directory containing your custom scripts.
try:
    sys.path.append(r"/home/lishaogang/NEU/validation-set")
    import anno_self
    print("Custom annotation module 'anno_self' loaded successfully.")
except ImportError:
    print("Warning: Custom annotation module 'anno_self' not found. Annotation steps will require manual implementation.")

In [4]:
# --- Set a global random seed for reproducibility ---
seed = 97
random.seed(seed)
np.random.seed(seed)

print(f"Global random seed set to: {seed}")

# --- Configure plotting parameters ---
sc.set_figure_params(frameon=False, fontsize=10, dpi=100)

In [16]:
adata.X = adata.layers['cp10k_counts'].copy()

In [18]:
import numpy as np
import pandas as pd

# 1. 从每个样本（orig.ident）中随机抽取一个细胞的索引
# 我们利用 groupby 分组后取每组的第一个 (head(1))，或者随机抽一个 (sample(1))
random_indices = adata.obs.groupby('orig.ident').sample(n=1, random_state=1).index

# 2. 提取这些细胞的表达量子集
subset_adata = adata[random_indices]
# 3. 计算每个提取细胞的基因表达总和
# 使用 np.ravel() 确保结果变成一维数组，无论 X 是稀疏还是稠密
row_sums = np.array(subset_adata.X.sum(axis=1)).ravel()

# 4. 创建验证结果表格
validation_df = pd.DataFrame({
    'Sample': subset_adata.obs['orig.ident'].values,
    'Cell_ID': subset_adata.obs_names,
    'Total_Expression': row_sums
})

# 5. 打印结果
print("抽样验证结果：")
print(validation_df)

# 6. 验证是否全部接近 10000
is_all_10000 = np.allclose(row_sums, 10000, atol=1e-2)
print(f"\n是否所有抽样细胞的和都为 10000: {is_all_10000}")

In [19]:
sc.pp.log1p(adata) 

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    batch_key='dataset',
    subset=False # 标记但不立即过滤
)

# --- 2c. 重新进行缩放、PCA 和批次校正 ---
print("步骤 2c: 重新进行缩放、PCA 和 Harmony 批次校正...")

# 仅在高变基因上进行后续计算
adata_hvg = adata[:, adata.var.highly_variable].copy()
sc.pp.scale(adata_hvg, max_value=10)
sc.tl.pca(adata_hvg, n_comps=60, random_state=seed)

# 重新运行 Harmony
# 使用一个新的 adjusted_basis 名称，以避免与旧结果混淆
sc.external.pp.harmony_integrate(
    adata_hvg,
    key='orig.ident',
    basis='X_pca',
    adjusted_basis='X_pca_harmony', # 使用新名称
    random_state=seed
)

# 将计算结果传回主对象
adata.obsm['X_pca'] = adata_hvg.obsm['X_pca']
adata.obsm['X_pca_harmony'] = adata_hvg.obsm['X_pca_harmony']
del adata_hvg
gc.collect()

In [25]:
adata

In [26]:
if 'pca' in adata.uns:
    sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)
else:
    print("错误：adata.uns 中没有找到 pca 统计信息。")

In [27]:
# 1. 针对高变基因重新运行 PCA
# use_highly_variable=True 会让 Scanpy 自动只看那 2000 个基因
# n_comps=50 足够我们观察趋势了
sc.tl.pca(adata, n_comps=50, use_highly_variable=True)

# 2. 现在可以画碎石图了
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

In [ ]:


# --- 2d. 重新计算邻接图、UMAP 和聚类 ---
print("步骤 2d: 重新计算邻接图、UMAP 和 Leiden 聚类...")
# 使用新的 Harmony 结果
sc.pp.neighbors(adata, n_pcs=15, use_rep='X_pca_harmony', random_state=seed)
sc.tl.umap(adata, random_state=seed)

# 使用新的 key_added，以避免覆盖旧的聚类结果
sc.tl.leiden(adata, resolution=2.0, key_added='leiden', random_state=seed)

# 可视化新的精细聚类结果
print("可视化新的精细聚类结果...")
sc.pl.umap(adata, color='leiden', legend_loc='on data', title='Refined Cell Clusters (After Filtering)')


In [28]:
sc.pl.umap(adata, color='leiden', legend_loc='on data', title='Refined Cell Clusters (After Filtering)')


In [ ]:


# --- 2d. 重新计算邻接图、UMAP 和聚类 ---
print("步骤 2d: 重新计算邻接图、UMAP 和 Leiden 聚类...")
# 使用新的 Harmony 结果
sc.pp.neighbors(adata, n_pcs=30, use_rep='X_pca_harmony', random_state=seed)
sc.tl.umap(adata, random_state=seed)

# 使用新的 key_added，以避免覆盖旧的聚类结果
sc.tl.leiden(adata, resolution=2.0, key_added='leiden_30pc', random_state=seed)

# 可视化新的精细聚类结果
print("可视化新的精细聚类结果...")
sc.pl.umap(adata, color='leiden_30pc', legend_loc='on data', title='Refined Cell Clusters (After Filtering)')

In [41]:
import matplotlib.pyplot as plt
import scanpy as sc

# 确保adata已加载
# adata = sc.read_h5ad('your_data.h5ad')

# 绘制多个color维度的UMAP（自动创建多面板）
# 1. 设置整体画布尺寸（ncols=2 让两个图并排，figsize控制整体大小）
fig, axes = plt.subplots(1, 2, figsize=(16, 8))  # 1行2列，总宽度16，高度8（保证每个子图8x8方形）

# 2. 分别绘制每个color维度，指定对应的ax
# 第一个面板：leiden_30pc
sc.pl.umap(adata, 
           color='leiden_30pc',
           legend_loc='on data', 
           title='Refined Cell Clusters',
           ax=axes[0],          # 指定第一个子图ax
           frameon=True, 
           show=False)

# 第二个面板：dataset
sc.pl.umap(adata, 
           color='dataset',
           legend_loc='on data', 
           title='Dataset Source',
           ax=axes[1],          # 指定第二个子图ax
           frameon=True, 
           show=False)

# 3. 强制每个子图都是正方形
for ax in axes:
    ax.set_box_aspect(1)

# 优化布局，避免重叠
plt.tight_layout()
plt.show()

In [59]:
#计算leiden_30pc中第28簇的每个dataset的细胞数
# 1. 筛选出leiden_30pc为28的细胞
cluster_28 = adata[adata.obs['leiden_30pc'] == '28']  # 注意：如果leiden_30pc是数值类型，去掉引号写成28

# 2. 统计该簇中每个dataset的细胞数量
count_by_dataset = cluster_28.obs['dataset'].value_counts()

# 3. 打印结果
print("leiden_30pc第28簇中各dataset的细胞数：")
print(count_by_dataset)

In [50]:
import matplotlib.pyplot as plt
import scanpy as sc

# 1. 手动创建 8x8 的方形画布
fig, ax = plt.subplots(figsize=(8, 11))

# 2. 绘图时传入 ax 参数
sc.pl.umap(adata, 
           color='dataset', 
           #legend_loc='on data', 
           title='Refined Cell Clusters (Square)',
           ax=ax,           # 关键：指定画布
           frameon=True, 
           show=False)      # show=False 方便后续微调

# 3. 强制让绘图区域变成绝对正方形（可选）
ax.set_box_aspect(1) 

plt.show()

In [60]:
adata_GSE186802 = adata[adata.obs['dataset'] == 'GSE186802'].copy()
adata_GSE244499 = adata[adata.obs['dataset'] == 'GSE244499'].copy()
adata_GSE133028 = adata[adata.obs['dataset'] == 'GSE133028'].copy()


In [46]:
import matplotlib.pyplot as plt
import scanpy as sc

# 1. 创建1行2列的画布（两个子图）
fig, axes = plt.subplots(1, 2, figsize=(16, 8))  

# 第一个面板：用orig.ident作为color，指定到axes[0]
sc.pl.umap(adata_GSE186802, 
           color='orig.ident',
           legend_loc='on data', 
           title='Orig. Ident Source',  # 区分标题
           ax=axes[0],  # 第一个子图位置
           frameon=True, 
           show=False)

# 第二个面板：用dataset作为color，指定到axes[1]
sc.pl.umap(adata_GSE186802, 
           color='dataset',
           legend_loc='on data', 
           title='Dataset Source',
           ax=axes[1],  # 第二个子图位置
           frameon=True, 
           show=False)

# 强制每个子图为正方形
for ax in axes:
    ax.set_box_aspect(1)

plt.tight_layout()
plt.show()

In [55]:
import matplotlib.pyplot as plt
import scanpy as sc

# 1. 创建1行2列的画布（两个子图）
fig, axes = plt.subplots(1, 2, figsize=(16, 8))  

# 第一个面板：用orig.ident作为color，指定到axes[0]
sc.pl.umap(adata_GSE244499, 
           color='orig.ident',
           legend_loc='on data', 
           title='Orig. Ident Source',  # 区分标题
           ax=axes[0],  # 第一个子图位置
           frameon=True, 
           show=False)

# 第二个面板：用dataset作为color，指定到axes[1]
sc.pl.umap(adata_GSE244499, 
           color='dataset',
           legend_loc='on data', 
           title='Dataset Source',
           ax=axes[1],  # 第二个子图位置
           frameon=True, 
           show=False)

# 强制每个子图为正方形
for ax in axes:
    ax.set_box_aspect(1)

plt.tight_layout()
plt.show()

In [61]:
import matplotlib.pyplot as plt
import scanpy as sc

# 1. 创建1行2列的画布（两个子图）
fig, axes = plt.subplots(1, 2, figsize=(16, 8))  

# 第一个面板：用orig.ident作为color，指定到axes[0]
sc.pl.umap(adata_GSE133028, 
           color='orig.ident',
           legend_loc='on data', 
           title='Orig. Ident Source',  # 区分标题
           ax=axes[0],  # 第一个子图位置
           frameon=True, 
           show=False)

# 第二个面板：用dataset作为color，指定到axes[1]
sc.pl.umap(adata_GSE133028, 
           color='dataset',
           legend_loc='on data', 
           title='Dataset Source',
           ax=axes[1],  # 第二个子图位置
           frameon=True, 
           show=False)

# 强制每个子图为正方形
for ax in axes:
    ax.set_box_aspect(1)

plt.tight_layout()
plt.show()

In [29]:
#保存 adata 到文件
adata.write('adata_filtered.h5ad', compression='gzip')

In [9]:
# --- Define a core list of marker genes for major CSF populations ---
all_marker_genes_list = [
    # T cells
    "CD3D", "CD3E", "CD3G",

    # B cell
    "MS4A1", "CD79A", "CD19",

    # Plasma cell
    # Note: For Plasma cell, we list the positive markers.
    # The down-regulation of B cell markers (MS4A1, CD19) is checked against the B cell markers.
    "IGHG1", "MZB1", "SDC1",

    # Myeloid cell (for Monocytes and conventional DCs)
    "LYZ", "S100A8", "S100A9", "VCAN",

    # Plasmacytoid Dendritic Cells (pDC)
    "LILRA4", "IL3RA", "CLEC4C",

    # B cell
    "NKG7", "FCGR3A", "FGFBP2", "CX3CR1",'SLC4A10',
    
    # NKT cells
    "KLRB1",'TRBC1','TRBC2','TRAV10','ZBTB16',
    
    #T cell
    "CD8A", "CD4",
    
    # Gamma-Delta T cell
    "TRGV9", "TRDV2",

    # Platelet Markers (Potential Contamination)
    "PPBP", "PF4", "GP9", "ITGA2B",

    # Erythrocyte Markers (Potential Contamination)
    "HBB", "HBA1", "HBA2",
]
print(f"定义的 Marker 基因总数: {len(all_marker_genes_list)}")

In [68]:
adata.raw = adata.copy()  # 保存原始数据

In [10]:


# --- Check for marker gene availability in the data ---
# Use .raw.var_names as it contains all genes before HVG filtering.
available_genes_in_raw = adata.raw.var_names.tolist()
genes_to_plot = [gene for gene in all_marker_genes_list if gene in available_genes_in_raw]

skipped_genes = set(all_marker_genes_list) - set(genes_to_plot)
if skipped_genes:
    print(f"Warning: The following genes were not found in the data and will be skipped:\n{list(skipped_genes)}")

In [46]:
import importlib
import anno_self
importlib.reload(anno_self)

In [73]:
# 1. 设置全局图像大小为 8x8
sc.set_figure_params(figsize=(8, 8), dpi=100)



In [43]:
# ==============================================================================
# 阶段三：对新的精细聚类进行重新注释
# ==============================================================================
print("\n--- 阶段三：对新的精细聚类进行重新注释 ---")

# --- 3a. 生成新的点图和总结表 ---
print("步骤 3a: 生成新的点图和总结表用于注释...")
# 注意 groupby 参数现在是 'leiden'
dotplot = sc.pl.dotplot(
    adata,
    genes_to_plot, # 使用之前定义好的基因列表
    groupby="leiden_30pc",
    standard_scale='var',
    use_raw=True,
    return_fig=True,
    show=False
)

mean_expression_df = dotplot.dot_color_df
fraction_df = dotplot.dot_size_df
final_summary_df = anno_self.generate_summary_table(mean_expression_df, fraction_df)

sc.pl.dotplot(
    adata,
    genes_to_plot, # 使用之前定义好的基因列表
    groupby="leiden_30pc",
    standard_scale='var',
    cmap='Reds',
    use_raw=True
)


summaries = anno_self.generate_cluster_summaries(final_summary_df)

# 使用 option_context 临时修改设置，不影响后续代码
with pd.option_context('display.max_colwidth', None):
    print(summaries)


summaries = anno_self.generate_cluster_summaries(final_summary_df)

# 使用 option_context 临时修改设置，不影响后续代码
with pd.option_context('display.max_colwidth', None):
    print(summaries)


# --- 3b. 执行自动化注释 ---
# 假设您的 anno_self.annotate_cluster 函数仍然适用
# 如果您有更精细的髓系注释函数（如 annotate_CSF_myeloid），也可以在这里使用
print("步骤 3b: 执行自动化注释...")
annotations_list = final_summary_df.apply(anno_self.annotate_cluster, axis=1) # 或者用你更具体的函数


# Convert the annotation result (a Series) into a DataFrame and set column names
# reset_index() moves the original index (cluster IDs) into a column named 'index'
# rename() changes the column names to 'Cluster' (for cluster IDs) and 'Cell Type' (for annotations)
annotations_df = annotations_list.reset_index().rename(columns={'index': 'Cluster', 0: 'Cell Type'})


# Generate an explanation column for the annotation DataFrame
# Apply the generate_explanation function from anno_self row-wise to annotations_df
# This function typically creates a text explanation for how each cluster's cell type was determined

annotations_df['Explanation'] = annotations_df.apply(
    lambda row: anno_self.generate_explanation(row, final_summary_df),  # 额外传入 final_summary_df
    axis=1
)
# Print the annotated DataFrame using the custom pretty_print_df function from anno_self
# This function usually formats the output for better readability (e.g., adjusting column widths)
anno_self.pretty_print_df(annotations_df)




# --- 3c. 将注释映射回 AnnData 对象 ---
print("步骤 3c: 将注释映射回 AnnData 对象...")
annotation_dict = annotations_df.set_index('Cluster')['Cell Type'].to_dict()

# 创建一个新的 obs 列来存储精细化的注释
adata.obs['cell_type'] = adata.obs['leiden_30pc'].map(annotation_dict).astype('category')





# 1. 创建1行2列的画布（两个子图）
fig, axes = plt.subplots(1, 2, figsize=(16, 8))  

# 第一个面板：用leiden_30pc作为color，指定到axes[0]
sc.pl.umap(adata, 
           color='leiden_30pc',
           legend_loc='on data', 
           title='leiden_30pc Source',  # 区分标题
           ax=axes[0],  # 第一个子图位置
           frameon=True, 
           show=False)

# 第二个面板：用cell_type作为color，指定到axes[1]
sc.pl.umap(adata, 
           color='cell_type',
           legend_loc='on data', 
           title='cell_type Source',
           ax=axes[1],  # 第二个子图位置
           frameon=True, 
           show=False)

# 强制每个子图为正方形
for ax in axes:
    ax.set_box_aspect(1)

plt.tight_layout()
plt.show()
    

In [151]:
#保存数据
adata.write_h5ad('NEU_anno_celltype.h5ad')

In [147]:
sc.pl.umap(adata, 
           color='PPBP',
           legend_loc='on data', 
           title='cell_type Source',
           frameon=False)

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc

# 1. 准备颜色字典：先给所有簇设为浅灰色，再给目标簇设为红色
target_clusters = ['19', '28', '35', '38']  # 目标簇列表
categories = adata.obs['leiden_30pc'].cat.categories
colors_dict = {cat: "#b1a9a9" for cat in categories}  # 所有簇默认浅灰色
for cluster in target_clusters:
    colors_dict[cluster] = 'red'  # 逐个给目标簇设红色

fig, ax = plt.subplots(figsize=(4, 4))

# 2. 绘制UMAP：指定显示目标簇，且红色点在最上层
sc.pl.umap(adata, 
           color='leiden_30pc', 
           groups=target_clusters,  # 只显示这4个簇（背景为NA）
           palette=colors_dict,     # 应用颜色字典
           ax=ax, 
           title='Focus on Cluster 19 (Optimized)',
           sort_order=True,         # 红色点画在最上层，避免遮挡
           na_color='#f0f0f0',      # 背景（其他簇）设为淡灰色
           frameon=True, 
           show=False)

ax.set_box_aspect(1)  # 保持正方形比例
plt.show()

In [13]:
#删除19,28，29,33,35，38，40
# 1. 定义要剔除的簇
clusters_to_remove = ['19', '21', '28', '29', '33', '35', '38', '40']

# 2. 别忘了你之前提到的 32 簇 (红细胞双细胞)，建议一起删掉
if '32' not in clusters_to_remove:
    clusters_to_remove.append('32')

# 3. 执行剔除并创建干净的对象
print(f"原始细胞数: {adata.n_obs}")
adata_clean = adata[~adata.obs['leiden_30pc'].isin(clusters_to_remove)].copy()
print(f"清洗后细胞数: {adata_clean.n_obs}")

# 4. 统计删除了多少
removed_count = adata.n_obs - adata_clean.n_obs
print(f"总计剔除了 {removed_count} 个噪音细胞 ({removed_count/adata.n_obs:.2%})")

In [14]:
adata_clean

In [ ]:
# 1. 检查清洗后的 UMAP
# size=0.5 因为点很多，调小一点可以看清密度

sc.pl.umap(adata_clean, color='leiden_30pc', 
           size=0.5, 
           title='Cleaned Global Atlas (639k cells)',
           legend_loc='on data',
           frameon=False)

In [15]:
sc.set_figure_params(figsize=(8, 8), dpi=100)
# 1. 检查清洗后的 UMAP
# size=0.5 因为点很多，调小一点可以看清密度

sc.pl.umap(adata_clean, color='cell_type', 

           title='Cleaned Global Atlas (639k cells)',
           legend_loc='on data',
           frameon=False)

In [170]:
import matplotlib.pyplot as plt
import scanpy as sc

# 确保adata已加载
# adata = sc.read_h5ad('your_data.h5ad')

# 绘制多个color维度的UMAP（自动创建多面板）
# 1. 设置整体画布尺寸（ncols=2 让两个图并排，figsize控制整体大小）
fig, axes = plt.subplots(1, 2, figsize=(12, 6))  # 1行2列，总宽度16，高度8（保证每个子图8x8方形）

# 2. 分别绘制每个color维度，指定对应的ax
# 第一个面板：cell_type
sc.pl.umap(adata_clean, 
           color='cell_type',
           legend_loc='on data', 
           title='Refined Cell Clusters',
           ax=axes[0],          # 指定第一个子图ax
           frameon=True, 
           show=False)

# 第二个面板：dataset
sc.pl.umap(adata_clean, 
           color='dataset',
           legend_loc='on data', 
           title='Dataset Source',
           ax=axes[1],          # 指定第二个子图ax
           frameon=True, 
           show=False)

# 3. 强制每个子图都是正方形
for ax in axes:
    ax.set_box_aspect(1)

# 优化布局，避免重叠
plt.tight_layout()
plt.show()

In [143]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# 设置绘图风格
sns.set_style("white") 

# ===================== 步骤1：数据准备 =====================
sample_metadata = adata.obs[['dataset', 'Disease', 'orig.ident']].drop_duplicates()
plot_data = sample_metadata.groupby(['dataset', 'Disease'], observed=False).size().unstack(fill_value=0)
disease_list = plot_data.columns.tolist()
totals = plot_data.sum(axis=1) # 计算每个柱子的总高度

# ===================== 步骤2：配色 =====================
# 使用更加柔和的配色方案
colors = plt.cm.get_cmap('tab20')(np.linspace(0, 1, len(disease_list)))

# ===================== 步骤3：绘图 =====================
fig, ax = plt.subplots(figsize=(16, 8), dpi=100)
bottom = np.zeros(len(plot_data.index))

for i, disease in enumerate(disease_list):
    values = plot_data[disease].values
    bars = ax.bar(
        plot_data.index, values, 
        bottom=bottom,
        label=disease,
        color=colors[i],
        edgecolor='white', # 柱子间使用白色描边，更有高级感
        linewidth=0.8
    )
    
    # 柱内标注疾病名
    current_bottom = bottom
    for j, val in enumerate(values):
        if val >= 2.5: # 仅在空间足够的情况下标注
            y_pos = current_bottom[j] + val / 2
            # 计算文字颜色：深色背景用白字，浅色用黑字
            brightness = np.mean(colors[i][:3])
            txt_color = 'white' if brightness < 0.6 else '#333333'
            
            # 缩短显示名称逻辑
            display_name = disease if len(disease) <= 12 else disease[:10] + ".."
            ax.text(
                j, y_pos, display_name,
                ha='center', va='center',
                color=txt_color,
                fontweight='bold', 
                fontsize=8
            )
    bottom += values

# ===================== 步骤4：在柱子顶部标注总数 =====================
for j, total in enumerate(totals):
    ax.text(
        j, total + 0.5, # 在顶部上方 0.5 个单位处
        f'n={int(total)}',
        ha='center', va='bottom',
        fontsize=10, fontweight='bold', color='#444444'
    )

# ===================== 步骤5：美化与留白 =====================
# 1. 增加顶部留白 (y轴范围扩大 15%)
ax.set_ylim(0, totals.max() * 1.15)

# 2. 去除顶部和右侧的边框
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# 3. 添加淡淡的水平网格线
ax.yaxis.grid(True, linestyle='--', alpha=0.6, color='#e0e0e0')
ax.set_axisbelow(True) # 确保网格线在柱子下方

# 4. 标题与轴标签
plt.title('Sample Composition Across Datasets', fontsize=18, pad=20, fontweight='bold')
plt.xlabel('Dataset Source', fontsize=14, labelpad=15)
plt.ylabel('Total Number of Samples', fontsize=14, labelpad=15)

# 5. 刻度美化
plt.xticks(rotation=40, ha='right', fontsize=11)
plt.yticks(fontsize=11)

# 6. 图例优化
plt.legend(
    title='$\mathbf{Disease\ Status}$', 
    bbox_to_anchor=(1.01, 1), 
    loc='upper left',
    frameon=False,
    fontsize=10
)

plt.tight_layout()
plt.show()

In [16]:
#保存adata_clean

adata_clean.write_h5ad('NEU_adata_clean.h5ad')

In [ ]:
import scanpy as sc
import scanpy.external as sce
import gc
import matplotlib.pyplot as plt
import numpy as np

# 设置随机种子
seed = 42

# --- 步骤 1: 筛选高变基因 (HVG) ---
print("正在计算高变基因...")
sc.pp.highly_variable_genes(
    adata_clean, 
    n_top_genes=2000, 
    batch_key='dataset', 
    flavor='seurat', 
    subset=False
)

# --- 步骤 2: 提取临时子集进行计算 ---
print("创建临时 HVG 对象用于计算...")
adata_hvg = adata_clean[:, adata_clean.var.highly_variable].copy()

# --- 步骤 3: 预处理 (Scale & PCA) ---
print("正在进行 Scale 和 PCA...")
sc.pp.scale(adata_hvg, max_value=10)
# 计算 50 个 PC 以便观察完整的方差下降趋势
sc.tl.pca(adata_hvg, n_comps=50, random_state=seed)

# --- 步骤 4: 重新运行 Harmony 批次校正 ---
print("正在运行 Harmony 整合 (基于 orig.ident)...")
sce.pp.harmony_integrate(
    adata_hvg, 
    key='orig.ident', 
    basis='X_pca', 
    adjusted_basis='X_pca_harmony',
    max_iter_harmony=20 
)

# --- 步骤 5: 将计算出的坐标和统计信息传回全基因对象 ---
print("将坐标和 PCA 统计信息映射回主对象...")
adata_clean.obsm['X_pca'] = adata_hvg.obsm['X_pca']
adata_clean.obsm['X_pca_harmony'] = adata_hvg.obsm['X_pca_harmony']
# 重要：传回 uns['pca'] 以便后续画碎石图
adata_clean.uns['pca'] = adata_hvg.uns['pca']


In [20]:
# --- 步骤 5: 将计算出的坐标和统计信息传回全基因对象 ---
print("将坐标和 PCA 统计信息映射回主对象...")
# 1. 映射细胞层面的PCA结果（这部分没问题，因为细胞数一致）
adata_clean.obsm['X_pca'] = adata_hvg.obsm['X_pca']
adata_clean.obsm['X_pca_harmony'] = adata_hvg.obsm['X_pca_harmony']
# 2. 映射PCA的方差解释等全局信息（这部分也没问题）
adata_clean.uns['pca'] = adata_hvg.uns['pca']

# 3. 正确映射基因层面的PCA载荷（varm['PCs']）
# 创建和adata_clean匹配的空矩阵，初始值为0
pca_loadings = np.zeros((adata_clean.n_vars, adata_hvg.obsm['X_pca'].shape[1]))
# 找到HVG在adata_clean中的索引
hvg_mask = adata_clean.var.highly_variable.values
# 只将HVG的PCA载荷填充到对应位置
pca_loadings[hvg_mask, :] = adata_hvg.varm['PCs']
# 赋值给主对象
adata_clean.varm['PCs'] = pca_loadings

# 释放内存
del adata_hvg, pca_loadings, hvg_mask
gc.collect()

In [23]:
# 检查PCA关键数据是否存在
print("检查PCA数据：")
print(f"是否有方差贡献率: {'variance_ratio' in adata_clean.uns['pca']}")
print(f"PCA方差贡献率长度: {len(adata_clean.uns['pca']['variance_ratio'])}")
# 重新映射（如果缺失）
if 'pca' not in adata_clean.uns:
    adata_clean.uns['pca'] = adata_hvg.uns['pca']  # 从HVG对象重新复制

In [3]:
dir()

In [29]:
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
import matplotlib  # 新增：导入matplotlib核心模块

# ===================== 关键：解决字体问题 =====================
# 1. 关闭字体查找的冗余日志
matplotlib.rcParams['font.family'] = 'DejaVu Sans'  # Linux系统默认自带的无衬线字体
matplotlib.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题
# 2. 禁用字体调试日志（彻底消除findfont提示）
import logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

# ===================== 绘制 PCA 碎石图 =====================
print("正在生成 PCA 碎石图...")
# 1. 绘制方差贡献率图
sc.pl.pca_variance_ratio(adata_clean, n_pcs=50, log=True, show=False)
plt.savefig('pca_variance_ratio.png')  # 自动保存一张图备查
plt.show()

# 2. 绘制累计方差图 (可选，能帮你更科学地看 90% 阈值)
var_ratio = adata_clean.uns['pca']['variance_ratio']
cum_var_ratio = np.cumsum(var_ratio)
plt.figure(figsize=(8, 3))
plt.plot(range(1, len(cum_var_ratio) + 1), cum_var_ratio, 'o-')
plt.axhline(y=0.9, color='r', linestyle='--')
plt.title('Cumulative Variance Explained')
plt.grid(alpha=0.3)
plt.savefig('pca_cumulative_variance.png')  # 新增：保存累计方差图
plt.show()

# 可选：如果想进一步清理matplotlib配置，运行后重置
plt.close('all')

In [5]:
adata_clean = sc.read_h5ad('/home/lishaogang/NEU_2026/NEU_adata_clean_refined.h5ad')

In [ ]:
# ===============================================================

# --- 步骤 6: 重新计算邻居图和 UMAP ---
# 这里使用了你确定的 40 个 PC
print("正在计算邻居图和 UMAP...")
sc.pp.neighbors(adata_clean, n_neighbors=30, n_pcs=20, use_rep='X_pca_harmony')
sc.tl.umap(adata_clean, min_dist=0.3, spread=1.0, random_state=seed)

# --- 步骤 7: 重新聚类 (Leiden) ---
print("正在重新聚类...")
sc.tl.leiden(adata_clean, resolution=0.8, key_added='leiden_refined', random_state=seed)



In [6]:
# ===============================================================

# --- 步骤 6: 重新计算邻居图和 UMAP ---
# 这里使用了你确定的 40 个 PC
print("正在计算邻居图和 UMAP...")
sc.pp.neighbors(adata_clean, n_neighbors=30, n_pcs=30, use_rep='X_pca_harmony')
sc.tl.umap(adata_clean, min_dist=0.3, spread=1.0, random_state=seed)

# --- 步骤 7: 重新聚类 (Leiden) ---
print("正在重新聚类...")
sc.tl.leiden(adata_clean, resolution=1.5, key_added='leiden_refined', random_state=seed)


In [17]:
import scanpy as sc
import matplotlib.pyplot as plt

# ===================== 解决核心报错 + 精准控制图片大小 + 缩小字体 =====================
print("正在生成最终优化图...")

# 全局设置字体大小（优先缩小所有matplotlib文字）
plt.rcParams['font.size'] = 8  # 全局默认字体大小（原默认10，改为8缩小）
plt.rcParams['axes.titlesize'] = 9  # 子图标题大小
plt.rcParams['axes.labelsize'] = 8  # 坐标轴标签大小
plt.rcParams['legend.fontsize'] = 7  # 图例字体大小
plt.rcParams['xtick.labelsize'] = 7  # x轴刻度字体
plt.rcParams['ytick.labelsize'] = 7  # y轴刻度字体

# 1. 手动创建画布和子图（尺寸保持6x3）
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(8, 4), sharex=True, sharey=True)

# 2. 绘制第一个UMAP（cell_type），指定图例字体大小
sc.pl.umap(
    adata_clean, 
    color='cell_type',
    size=1, 
    alpha=0.6,
    legend_loc='on data',
    frameon=False,
    show=False,
    title='Refined Clusters (Cleaned)',
    ax=ax1,
    legend_fontsize=7  # 单独指定图例字体（优先级高于全局）
)

# 3. 绘制第二个UMAP（dataset）
sc.pl.umap(
    adata_clean, 
    color='leiden_refined',
    size=1, 
    alpha=0.6,
    legend_loc='on data',
    frameon=False,
    show=False,
    title='Batch Integration Check',
    ax=ax2,
    legend_fontsize=7
)

# 4. 优化布局（水平排列无需调整hspace，改为调整wspace）
plt.tight_layout()
plt.subplots_adjust(wspace=0.1)  # 水平间距（原hspace是垂直，这里改wspace）

# 5. 保存并显示
plt.savefig('umap_final.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

# 重置全局字体设置（避免影响后续绘图）
plt.rcParams.update(plt.rcParamsDefault)

In [41]:
# 1. 检查清洗后的 UMAP
# size=0.5 因为点很多，调小一点可以看清密度

sc.pl.umap(adata_clean, color=['leiden_refined', 'leiden_30pc'], 
           size=0.5, 
           title='Cleaned Global Atlas (639k cells)',
           legend_loc='on data',
           frameon=False)

In [19]:
# ==============================================================================
# 阶段三：对新的精细聚类进行重新注释
# ==============================================================================
print("\n--- 阶段三：对新的精细聚类进行重新注释 ---")

# --- 3a. 生成新的点图和总结表 ---
print("步骤 3a: 生成新的点图和总结表用于注释...")
# 注意 groupby 参数现在是 'leiden'
dotplot = sc.pl.dotplot(
    adata_clean,
    genes_to_plot, # 使用之前定义好的基因列表
    groupby="leiden_refined",
    standard_scale='var',
    use_raw=True,
    return_fig=True,
    show=False
)

mean_expression_df = dotplot.dot_color_df
fraction_df = dotplot.dot_size_df
final_summary_df = anno_self.generate_summary_table(mean_expression_df, fraction_df)

sc.pl.dotplot(
    adata_clean,
    genes_to_plot, # 使用之前定义好的基因列表
    groupby="leiden_refined",
    standard_scale='var',
    cmap='Reds',
    use_raw=True
)


summaries = anno_self.generate_cluster_summaries(final_summary_df)

# 使用 option_context 临时修改设置，不影响后续代码
with pd.option_context('display.max_colwidth', None):
    print(summaries)


summaries = anno_self.generate_cluster_summaries(final_summary_df)

# 使用 option_context 临时修改设置，不影响后续代码
with pd.option_context('display.max_colwidth', None):
    print(summaries)


# --- 3b. 执行自动化注释 ---
# 假设您的 anno_self.annotate_cluster 函数仍然适用
# 如果您有更精细的髓系注释函数（如 annotate_CSF_myeloid），也可以在这里使用
print("步骤 3b: 执行自动化注释...")
annotations_list = final_summary_df.apply(anno_self.annotate_cluster, axis=1) # 或者用你更具体的函数


# Convert the annotation result (a Series) into a DataFrame and set column names
# reset_index() moves the original index (cluster IDs) into a column named 'index'
# rename() changes the column names to 'Cluster' (for cluster IDs) and 'Cell Type' (for annotations)
annotations_df = annotations_list.reset_index().rename(columns={'index': 'Cluster', 0: 'Cell Type'})


# Generate an explanation column for the annotation DataFrame
# Apply the generate_explanation function from anno_self row-wise to annotations_df
# This function typically creates a text explanation for how each cluster's cell type was determined

annotations_df['Explanation'] = annotations_df.apply(
    lambda row: anno_self.generate_explanation(row, final_summary_df),  # 额外传入 final_summary_df
    axis=1
)
# Print the annotated DataFrame using the custom pretty_print_df function from anno_self
# This function usually formats the output for better readability (e.g., adjusting column widths)
anno_self.pretty_print_df(annotations_df)




# --- 3c. 将注释映射回 AnnData 对象 ---
print("步骤 3c: 将注释映射回 AnnData 对象...")
annotation_dict = annotations_df.set_index('Cluster')['Cell Type'].to_dict()

# 创建一个新的 obs 列来存储精细化的注释
adata_clean.obs['cell_type'] = adata_clean.obs['leiden_refined'].map(annotation_dict).astype('category')





# 1. 创建1行2列的画布（两个子图）
fig, axes = plt.subplots(1, 2, figsize=(10,5))  

# 第一个面板：用leiden_refined作为color，指定到axes[0]
sc.pl.umap(adata_clean, 
           color='leiden_refined',
           legend_loc='on data', 
           title='leiden_refined Source',  # 区分标题
           ax=axes[0],  # 第一个子图位置
           frameon=False, 
           show=False)

# 第二个面板：用cell_type作为color，指定到axes[1]
sc.pl.umap(adata_clean, 
           color='cell_type',
           legend_loc='on data', 
           title='cell_type Source',
           ax=axes[1],  # 第二个子图位置
           frameon=False, 
           show=False)

# 强制每个子图为正方形
for ax in axes:
    ax.set_box_aspect(1)

plt.tight_layout()
plt.show()
    

In [ ]:
#删除25,26，27，28，29
adata_clean1 = adata_clean[~adata_clean.obs['leiden_refined'].isin([25, 26, 27, 28, 29]), :].copy()


In [ ]:
sc.tl.umap(adata_clean1, min_dist=0.3, spread=1.0, random_state=seed)

# --- 步骤 7: 重新聚类 (Leiden) ---
print("正在重新聚类...")
sc.tl.leiden(adata_clean1, resolution=1.5, key_added='leiden_refined1', random_state=seed)

# 1. 创建1行2列的画布（两个子图）
fig, axes = plt.subplots(1, 2, figsize=(12, 6))  

# 第一个面板：用leiden_refined1作为color，指定到axes[0]
sc.pl.umap(adata_clean1, 
           color='leiden_refined1',
           legend_loc='on data', 
           title='leiden_refined1 Source',  # 区分标题
           ax=axes[0],  # 第一个子图位置
           frameon=True, 
           show=False)

# 第二个面板：用cell_type作为color，指定到axes[1]
sc.pl.umap(adata_clean, 
           color='cell_type',
           legend_loc='on data', 
           title='cell_type Source',
           ax=axes[1],  # 第二个子图位置
           frameon=True, 
           show=False)

# 强制每个子图为正方形
for ax in axes:
    ax.set_box_aspect(1)

plt.tight_layout()
plt.show()
    


In [ ]:
# 1. 创建1行2列的画布（两个子图）
fig, axes = plt.subplots(1, 2, figsize=(12, 6))  

# 第一个面板：用leiden_refined作为color，指定到axes[0]
sc.pl.umap(adata_clean, 
           color='leiden_refined',
           legend_loc='on data', 
           title='leiden_refined Source',  # 区分标题
           ax=axes[0],  # 第一个子图位置
           frameon=True, 
           show=False)

# 第二个面板：用cell_type作为color，指定到axes[1]
sc.pl.umap(adata_clean, 
           color='cell_type',
           legend_loc='on data', 
           title='cell_type Source',
           ax=axes[1],  # 第二个子图位置
           frameon=True, 
           show=False)

# 强制每个子图为正方形
for ax in axes:
    ax.set_box_aspect(1)

plt.tight_layout()
plt.show()
    

In [50]:
# ==============================================================================
# 阶段三：对新的精细聚类进行重新注释
# ==============================================================================
print("\n--- 阶段三：对新的精细聚类进行重新注释 ---")

# --- 3a. 生成新的点图和总结表 ---
print("步骤 3a: 生成新的点图和总结表用于注释...")
# 注意 groupby 参数现在是 'leiden'
dotplot = sc.pl.dotplot(
    adata_clean,
    genes_to_plot, # 使用之前定义好的基因列表
    groupby="leiden_30pc",
    standard_scale='var',
    use_raw=True,
    return_fig=True,
    show=False
)

mean_expression_df = dotplot.dot_color_df
fraction_df = dotplot.dot_size_df
final_summary_df = anno_self.generate_summary_table(mean_expression_df, fraction_df)

sc.pl.dotplot(
    adata_clean,
    genes_to_plot, # 使用之前定义好的基因列表
    groupby="leiden_30pc",
    standard_scale='var',
    cmap='Reds',
    use_raw=True
)


summaries = anno_self.generate_cluster_summaries(final_summary_df)

# 使用 option_context 临时修改设置，不影响后续代码
with pd.option_context('display.max_colwidth', None):
    print(summaries)


summaries = anno_self.generate_cluster_summaries(final_summary_df)

# 使用 option_context 临时修改设置，不影响后续代码
with pd.option_context('display.max_colwidth', None):
    print(summaries)


# --- 3b. 执行自动化注释 ---
# 假设您的 anno_self.annotate_cluster 函数仍然适用
# 如果您有更精细的髓系注释函数（如 annotate_CSF_myeloid），也可以在这里使用
print("步骤 3b: 执行自动化注释...")
annotations_list = final_summary_df.apply(anno_self.annotate_cluster, axis=1) # 或者用你更具体的函数


# Convert the annotation result (a Series) into a DataFrame and set column names
# reset_index() moves the original index (cluster IDs) into a column named 'index'
# rename() changes the column names to 'Cluster' (for cluster IDs) and 'Cell Type' (for annotations)
annotations_df = annotations_list.reset_index().rename(columns={'index': 'Cluster', 0: 'Cell Type'})


# Generate an explanation column for the annotation DataFrame
# Apply the generate_explanation function from anno_self row-wise to annotations_df
# This function typically creates a text explanation for how each cluster's cell type was determined

annotations_df['Explanation'] = annotations_df.apply(
    lambda row: anno_self.generate_explanation(row, final_summary_df),  # 额外传入 final_summary_df
    axis=1
)
# Print the annotated DataFrame using the custom pretty_print_df function from anno_self
# This function usually formats the output for better readability (e.g., adjusting column widths)
anno_self.pretty_print_df(annotations_df)




# --- 3c. 将注释映射回 AnnData 对象 ---
print("步骤 3c: 将注释映射回 AnnData 对象...")
annotation_dict = annotations_df.set_index('Cluster')['Cell Type'].to_dict()

# 创建一个新的 obs 列来存储精细化的注释
adata_clean.obs['cell_type'] = adata_clean.obs['leiden_30pc'].map(annotation_dict).astype('category')





# 1. 创建1行2列的画布（两个子图）
fig, axes = plt.subplots(1, 2, figsize=(16, 8))  

# 第一个面板：用leiden_30pc作为color，指定到axes[0]
sc.pl.umap(adata_clean, 
           color='leiden_30pc',
           legend_loc='on data', 
           title='leiden_30pc Source',  # 区分标题
           ax=axes[0],  # 第一个子图位置
           frameon=True, 
           show=False)

# 第二个面板：用cell_type作为color，指定到axes[1]
sc.pl.umap(adata_clean, 
           color='cell_type',
           legend_loc='on data', 
           title='cell_type Source',
           ax=axes[1],  # 第二个子图位置
           frameon=True, 
           show=False)

# 强制每个子图为正方形
for ax in axes:
    ax.set_box_aspect(1)

plt.tight_layout()
plt.show()
    

In [49]:
import scanpy as sc
import matplotlib.pyplot as plt

# ===================== 解决核心报错 + 精准控制图片大小 + 缩小字体 =====================
print("正在生成最终优化图...")

# 全局设置字体大小（优先缩小所有matplotlib文字）
plt.rcParams['font.size'] = 8  # 全局默认字体大小（原默认10，改为8缩小）
plt.rcParams['axes.titlesize'] = 9  # 子图标题大小
plt.rcParams['axes.labelsize'] = 8  # 坐标轴标签大小
plt.rcParams['legend.fontsize'] = 7  # 图例字体大小
plt.rcParams['xtick.labelsize'] = 7  # x轴刻度字体
plt.rcParams['ytick.labelsize'] = 7  # y轴刻度字体

# 1. 手动创建画布和子图（尺寸保持6x3）
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(6, 3), sharex=True, sharey=True)

# 2. 绘制第一个UMAP（cell_type），指定图例字体大小
sc.pl.umap(
    adata_clean, 
    color='cell_type',
    size=1, 
    alpha=0.6,
    legend_loc='on data',
    frameon=False,
    show=False,
    title='Refined Clusters (Cleaned)',
    ax=ax1,
    legend_fontsize=7  # 单独指定图例字体（优先级高于全局）
)

# 3. 绘制第二个UMAP（dataset）
sc.pl.umap(
    adata_clean, 
    color='dataset',
    size=1, 
    alpha=0.6,
    legend_loc='on data',
    frameon=False,
    show=False,
    title='Batch Integration Check',
    ax=ax2,
    legend_fontsize=7
)

# 4. 优化布局（水平排列无需调整hspace，改为调整wspace）
plt.tight_layout()
plt.subplots_adjust(wspace=0.1)  # 水平间距（原hspace是垂直，这里改wspace）

# 5. 保存并显示
plt.savefig('umap_final.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

# 重置全局字体设置（避免影响后续绘图）
plt.rcParams.update(plt.rcParamsDefault)

In [11]:

# --- Check for marker gene availability in the data ---
# Use .raw.var_names as it contains all genes before HVG filtering.
available_genes_in_raw = adata_clean.raw.var_names.tolist()
genes_to_plot = [gene for gene in all_marker_genes_list if gene in available_genes_in_raw]

skipped_genes = set(all_marker_genes_list) - set(genes_to_plot)
if skipped_genes:
    print(f"Warning: The following genes were not found in the data and will be skipped:\n{list(skipped_genes)}")

In [34]:
adata_clean.write_h5ad('NEU_adata_clean_refined.h5ad')

In [39]:
#将'leiden'= 28 对应的细胞的cell_subtype列改为Myeloid cell
# 1. 将cell_type列转换为字符串类型
adata.obs['cell_type'] = adata.obs['cell_type'].astype(str)

# 2. 再执行赋值操作（此时无报错）
adata.obs.loc[adata.obs['leiden'] == '28', 'cell_type'] = 'Myeloid cell lineage'

#将cell_type列转换回分类类型
adata.obs['cell_type'] = adata.obs['cell_type'].astype('category')

In [40]:
# --- 3d. 最终可视化 ---
print("步骤 3d: 可视化最终的精细注释结果...")
sc.pl.umap(
    adata, 
    color='cell_type',
    frameon=False,
    title='Refined Cell Subtypes (After Filtering)',
    legend_loc='on data',
    legend_fontsize=8
)
plt.show()

In [42]:
# --- 3d. 最终可视化 ---
print("步骤 3d: 可视化最终的精细注释结果...")
sc.pl.umap(
    adata, 
    color='batch',
    frameon=False,
    title='Refined Cell Subtypes (After Filtering)',
    #legend_loc='on data',
    legend_fontsize=8
)
plt.show()

In [43]:
adata

In [45]:
adata.obs

In [ ]:
# 获取唯一的 disease 和 orig.ident 对应关系
source_cell_disease = adata.obs[['orig.ident', 'disease']].drop_duplicates()

# 按 disease 分组，并为每个组内的样本分配序号（明确设置 observed=False 以避免警告）
source_cell_disease['sample_number'] = source_cell_disease.groupby('disease', observed=False).cumcount() + 1

# 格式化序号为两位数
source_cell_disease['sample_number'] = source_cell_disease['sample_number'].apply(lambda x: f"{x:02d}")

# 创建 sample_ID 列（将 Categorical 类型转换为字符串）
source_cell_disease['sample_ID'] = source_cell_disease['disease'].astype(str) + '_' + source_cell_disease['sample_number'] + '_Discovery'

# 只保留需要映射的列
id_mapping = source_cell_disease[['orig.ident', 'sample_ID']]

# 将 sample_ID 映射回 adata.obs
adata.obs = adata.obs.merge(id_mapping, on='orig.ident', how='left')

# 查看结果示例
print(adata.obs[['orig.ident', 'disease', 'sample_ID']].head())

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 步骤1：对原始adata的X进行逆log1p转换（还原为线性归一化表达量）
# 注意：仅当adata.X是log1p处理后的结果时使用expm1
adata_inverse = adata.copy()
adata_inverse.X = np.expm1(adata_inverse.X)  # 逆log1p核心操作，还原线性表达量

# 步骤2：构建伪bulk数据（优先求和聚合，生物学意义更优）
# 提取表达量+sample_ID+batch+disease
obs_df = sc.get.obs_df(adata_inverse, keys=adata_inverse.var_names.tolist() + ['sample_ID', 'batch', 'disease'])

# 基因表达量：按sample_ID求和（方案B支持求和，更优）
gene_pseudobulk = obs_df.groupby('sample_ID')[adata_inverse.var_names].sum().reset_index()

# 提取样本元信息
sample_meta = obs_df.groupby('sample_ID').agg(
    batch=('batch', 'first'),
    disease=('disease', 'first')
).reset_index()

# 合并数据
pseudobulk_df = pd.merge(gene_pseudobulk, sample_meta, on='sample_ID', how='inner')

# 步骤3：构建伪bulk AnnData
pseudobulk_X = pseudobulk_df[adata_inverse.var_names].values
pseudobulk_obs = pseudobulk_df[['sample_ID', 'batch', 'disease']].set_index('sample_ID')
pseudobulk_var = adata_inverse.var.copy()
pseudobulk_adata = sc.AnnData(X=pseudobulk_X, obs=pseudobulk_obs, var=pseudobulk_var)

# 步骤4：伪bulk预处理（仅做1e4归一化，不再次log1p！）
sc.pp.normalize_total(pseudobulk_adata, target_sum=1e4)  # 仅线性归一化，无失真
sc.pp.scale(pseudobulk_adata, zero_center=True, max_value=10)  # PCA必需

# 步骤5：PCA计算+可视化（同之前流程，绘制两张图）
sc.tl.pca(pseudobulk_adata, svd_solver='arpack')



In [103]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# （前提：已完成pseudobulk_adata构建+PCA计算）

# 1. 核心：通过figsize设置图片整体大小
# 示例1：常规宽屏（适合展示，宽20英寸，高7英寸）
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 7))
# 示例2：更大尺寸（适合论文高清排版，宽24英寸，高8英寸）
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 8))
# 示例3：紧凑尺寸（适合插入PPT，宽16英寸，高6英寸）
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))  # 可直接修改括号内数值调整大小

# ---------------------- 左图：Batch分组 ----------------------
sc.pl.pca(
    pseudobulk_adata,
    color='batch',
    size=300,
    alpha=0.9,
    legend_loc='upper right',
    title='Pseudobulk PCA (Batch)',
    show=False,
    ax=ax1,
    label='sample_ID',
    frameon=True,
    legend_fontsize=10,
    palette=None
)
# 过滤图例
handles1, labels1 = ax1.get_legend_handles_labels()
valid1 = [(h, l) for h, l in zip(handles1, labels1) if l != 'sample_ID']
if valid1:
    ax1.legend(*zip(*valid1), loc='upper right', fontsize=10)
else:
    ax1.legend().remove()
# 缩短X轴距离
x_min, x_max = ax1.get_xlim()
new_x_min = x_min + (x_max - x_min) * 0.1
new_x_max = x_max - (x_max - x_min) * 0.1
ax1.set_xlim(new_x_min, new_x_max)

# ---------------------- 右图：Disease分组 ----------------------
sc.pl.pca(
    pseudobulk_adata,
    color='disease',
    size=300,
    alpha=0.9,
    legend_loc='upper right',
    title='Pseudobulk PCA (Disease)',
    show=False,
    ax=ax2,
    label='sample_ID',
    frameon=True,
    legend_fontsize=10,
    palette=None
)
# 过滤图例
handles2, labels2 = ax2.get_legend_handles_labels()
valid2 = [(h, l) for h, l in zip(handles2, labels2) if l != 'sample_ID']
if valid2:
    ax2.legend(*zip(*valid2), loc='upper right', fontsize=10)
else:
    ax2.legend().remove()
# 缩短X轴距离
x_min, x_max = ax2.get_xlim()
new_x_min = x_min + (x_max - x_min) * 0.1
new_x_max = x_max - (x_max - x_min) * 0.1
ax2.set_xlim(new_x_min, new_x_max)

# 2. 配套布局调整（不改变图片大小，优化内部排版）
plt.subplots_adjust(wspace=0.2)  # 调整子图间距（横向），不影响图片整体大小
plt.tight_layout()  # 自动调整内部元素，避免标签/图例重叠

# 3. 保存图片（dpi参数影响图片分辨率，不改变尺寸比例）
plt.savefig(
    'pseudobulk_pca_custom_size.png',
    dpi=300,  # 分辨率（每英寸像素数），dpi越大图片越清晰，文件越大
    bbox_inches='tight'  # 裁剪多余白边，不改变图片核心大小
)
plt.show()

In [104]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# 需先安装harmony-pytorch：pip install harmony-pytorch
from harmony import harmonize

# ---------------------- 1. 先完成伪bulk数据构建（复用你已有的代码） ----------------------
# 步骤1：逆log1p还原线性表达量
adata_inverse = adata.copy()
adata_inverse.X = np.expm1(adata_inverse.X)

# 步骤2：构建伪bulk（求和聚合）
obs_df = sc.get.obs_df(adata_inverse, keys=adata_inverse.var_names.tolist() + ['sample_ID', 'batch', 'disease'])
gene_pseudobulk = obs_df.groupby('sample_ID')[adata_inverse.var_names].sum().reset_index()
sample_meta = obs_df.groupby('sample_ID').agg(batch=('batch', 'first'), disease=('disease', 'first')).reset_index()
pseudobulk_df = pd.merge(gene_pseudobulk, sample_meta, on='sample_ID', how='inner')

# 步骤3：构建pseudobulk_adata
pseudobulk_X = pseudobulk_df[adata_inverse.var_names].values
pseudobulk_obs = pseudobulk_df[['sample_ID', 'batch', 'disease']].set_index('sample_ID')
pseudobulk_var = adata_inverse.var.copy()
pseudobulk_adata = sc.AnnData(X=pseudobulk_X, obs=pseudobulk_obs, var=pseudobulk_var)

# 步骤4：伪bulk预处理
sc.pp.normalize_total(pseudobulk_adata, target_sum=1e4)
sc.pp.scale(pseudobulk_adata, zero_center=True, max_value=10)

# 步骤5：先执行PCA（Harmony需要基于PCA结果去批次）
sc.tl.pca(pseudobulk_adata, svd_solver='arpack')


# ---------------------- 2. 使用Harmony对伪bulk的PCA结果去批次 ----------------------
# Harmony输入：PCA坐标 + 批次列（batch）
# 输出：校正后的PCA坐标，存储到.obsm['X_pca_harmony']
pseudobulk_adata.obsm['X_pca_harmony'] = harmonize(
    pseudobulk_adata.obsm['X_pca'],  # 原始PCA坐标
    pseudobulk_adata.obs,            # 样本元信息
    'batch'                          # 批次列名
)


# ---------------------- 3. 基于Harmony校正后的坐标绘制PCA图（并列Batch+Disease） ----------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

# 左图：Batch分组（基于Harmony校正后的PCA）
sc.pl.embedding(
    pseudobulk_adata,
    basis='pca_harmony',  # 核心：使用Harmony校正后的坐标
    color='batch',
    size=300,
    alpha=0.9,
    legend_loc='upper right',
    title='Pseudobulk PCA (After Harmony Correction, Batch)',
    show=False,
    ax=ax1,
    frameon=True,
    legend_fontsize=10
)
# 过滤图例
handles1, labels1 = ax1.get_legend_handles_labels()
valid1 = [(h, l) for h, l in zip(handles1, labels1) if l != 'sample_ID']
if valid1:
    ax1.legend(*zip(*valid1), loc='upper right', fontsize=10)
else:
    ax1.legend().remove()
# 调整X轴范围
x_min, x_max = ax1.get_xlim()
ax1.set_xlim(x_min + (x_max-x_min)*0.1, x_max - (x_max-x_min)*0.1)


# 右图：Disease分组（基于Harmony校正后的PCA）
sc.pl.embedding(
    pseudobulk_adata,
    basis='pca_harmony',  # 核心：使用Harmony校正后的坐标
    color='disease',
    size=300,
    alpha=0.9,
    legend_loc='upper right',
    title='Pseudobulk PCA (After Harmony Correction, Disease)',
    show=False,
    ax=ax2,
    frameon=True,
    legend_fontsize=10
)
# 过滤图例
handles2, labels2 = ax2.get_legend_handles_labels()
valid2 = [(h, l) for h, l in zip(handles2, labels2) if l != 'sample_ID']
if valid2:
    ax2.legend(*zip(*valid2), loc='upper right', fontsize=10)
else:
    ax2.legend().remove()
# 调整X轴范围
x_min, x_max = ax2.get_xlim()
ax2.set_xlim(x_min + (x_max-x_min)*0.1, x_max - (x_max-x_min)*0.1)


# 布局调整+保存
plt.subplots_adjust(wspace=0.2)
plt.tight_layout()
plt.savefig('pseudobulk_pca_harmony_corrected.png', dpi=300, bbox_inches='tight')
plt.show()

In [108]:
import pandas as pd

# 提取元数据
meta = pseudobulk_adata.obs[['batch', 'disease']]

# 生成交叉表
count_table = pd.crosstab(meta['batch'], meta['disease'])

print("--- 样本数量交叉表 ---")
print(count_table)

# 如果想看比例（每行总和为1），观察分布是否倾斜
prop_table = pd.crosstab(meta['batch'], meta['disease'], normalize='index')
print("\n--- 样本比例分布表 ---")
print(prop_table)

In [109]:
import matplotlib.pyplot as plt

# 绘图
prop_table.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='tab20')
plt.title('Disease distribution across Batches')
plt.ylabel('Proportion')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [110]:
from scipy.stats import chi2_contingency

chi2, p, dof, expected = chi2_contingency(count_table)
print(f"卡方检验 P值: {p}")

if p < 0.05:
    print("警告：批次与疾病分布存在显著相关性（存在混淆风险）。")
else:
    print("批次与疾病分布相对独立（实验设计较好）。")

In [107]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------------------- 1. 伪bulk数据构建（消除FutureWarning） ----------------------
# 步骤1：逆log1p还原线性表达量
adata_inverse = adata.copy()
adata_inverse.X = np.expm1(adata_inverse.X)

# 步骤2：构建伪bulk数据（添加observed=True消除警告）
obs_df = sc.get.obs_df(
    adata_inverse, 
    keys=adata_inverse.var_names.tolist() + ['sample_ID', 'batch', 'disease']
)

# 消除FutureWarning：添加observed=True
gene_pseudobulk = obs_df.groupby('sample_ID', observed=True)[adata_inverse.var_names].sum().reset_index()
sample_meta = obs_df.groupby('sample_ID', observed=True).agg(
    batch=('batch', 'first'),
    disease=('disease', 'first')
).reset_index()

# 合并数据
pseudobulk_df = pd.merge(gene_pseudobulk, sample_meta, on='sample_ID', how='inner')

# 步骤3：构建伪bulk AnnData
pseudobulk_X = pseudobulk_df[adata_inverse.var_names].values
pseudobulk_obs = pseudobulk_df[['sample_ID', 'batch', 'disease']].set_index('sample_ID')
pseudobulk_var = adata_inverse.var.copy()
pseudobulk_adata = sc.AnnData(X=pseudobulk_X, obs=pseudobulk_obs, var=pseudobulk_var)

# 步骤4：伪bulk预处理
sc.pp.normalize_total(pseudobulk_adata, target_sum=1e4)
sc.pp.scale(pseudobulk_adata, zero_center=True, max_value=10)

# 步骤5：先执行PCA（Harmony必须依赖原始PCA结果）
sc.tl.pca(pseudobulk_adata, svd_solver='arpack')

# ---------------------- 2. Scanpy官方Harmony去批次（彻底修正参数，核心！） ----------------------
# 正确写法1：纯位置参数（最简单，无歧义，推荐！）
# 格式：sc.external.pp.harmony_integrate(adata, 批次列名)
sc.external.pp.harmony_integrate(pseudobulk_adata, 'batch')

# 正确写法2：关键字参数（更清晰，可明确指定所有参数）
# sc.external.pp.harmony_integrate(
#     adata=pseudobulk_adata,
#     basis='X_pca',  # 原始PCA坐标（obsm['X_pca']）
#     adjusted_basis='X_pca_harmony',  # 校正后坐标存储位置
#     key='batch'  # 批次列名（部分Scanpy版本用key指定批次列，兼容写法）
# )

# ---------------------- 3. 绘制Harmony校正后的并列PCA图（样式不变） ----------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

# 左图：Batch分组（校正后）
sc.pl.embedding(
    pseudobulk_adata,
    basis='pca_harmony',  # 使用Harmony校正后的PCA坐标
    color='batch',
    size=300,
    alpha=0.9,
    legend_loc='upper right',
    title='Pseudobulk PCA (After Harmony Correction, Batch)',
    show=False,
    ax=ax1,
    frameon=True,
    legend_fontsize=10
)
# 过滤多余sample_ID图例
handles1, labels1 = ax1.get_legend_handles_labels()
valid1 = [(h, l) for h, l in zip(handles1, labels1) if l != 'sample_ID']
if valid1:
    ax1.legend(*zip(*valid1), loc='upper right', fontsize=10)
else:
    ax1.legend().remove()
# 缩短X轴距离
x_min, x_max = ax1.get_xlim()
new_x_min = x_min + (x_max - x_min) * 0.1
new_x_max = x_max - (x_max - x_min) * 0.1
ax1.set_xlim(new_x_min, new_x_max)

# 右图：Disease分组（校正后）
sc.pl.embedding(
    pseudobulk_adata,
    basis='pca_harmony',
    color='disease',
    size=300,
    alpha=0.9,
    legend_loc='upper right',
    title='Pseudobulk PCA (After Harmony Correction, Disease)',
    show=False,
    ax=ax2,
    frameon=True,
    legend_fontsize=10
)
# 过滤多余sample_ID图例
handles2, labels2 = ax2.get_legend_handles_labels()
valid2 = [(h, l) for h, l in zip(handles2, labels2) if l != 'sample_ID']
if valid2:
    ax2.legend(*zip(*valid2), loc='upper right', fontsize=10)
else:
    ax2.legend().remove()
# 缩短X轴距离
x_min, x_max = ax2.get_xlim()
new_x_min = x_min + (x_max - x_min) * 0.1
new_x_max = x_max - (x_max - x_min) * 0.1
ax2.set_xlim(new_x_min, new_x_max)

# 布局优化+保存高清图片
plt.subplots_adjust(wspace=0.2)
plt.tight_layout()
plt.savefig('pseudobulk_pca_harmony_final_no_error.png', dpi=300, bbox_inches='tight')
plt.show()

In [57]:
adata.raw = adata.copy()

In [111]:
adata.obs.head(1).to_dict()

In [114]:
adata

In [ ]:
# --- Subset Myeloid cells for re-analysis ---
adata_myeloid = adata[adata.obs['cell_type'] == 'Myeloid cell lineage'].copy()
print(f"Successfully extracted {adata_myeloid.n_obs} Myeloid cells for sub-clustering.")
gc.collect()

sc.pp.filter_genes(adata_myeloid, min_cells=1)


In [ ]:
# --- 8b. Re-identify HVGs within the myeloid subset ---
# This finds genes that are most variable specifically among myeloid cells.
print("\n--- Finding HVGs within the myeloid subset ---")
sc.pp.highly_variable_genes(adata_myeloid, batch_key='orig.ident', n_top_genes=2000)

# --- 8c. Re-run scaling, PCA, and batch correction ---
# Note: these operations are performed only on the newly identified HVGs.
adata_myeloid_hvg = adata_myeloid[:, adata_myeloid.var.highly_variable].copy()
sc.pp.scale(adata_myeloid_hvg, max_value=10)
sc.tl.pca(adata_myeloid_hvg, n_comps=50, random_state=seed)


# --- Drawing a Scree Plot ---
print("Drawing a scree plot to assist in determining the selection of n_pcs...")

# 1. Modify Matplotlib's global default figure size before plotting
#    This will affect all figures created afterwards
plt.rcParams['figure.figsize'] = (8, 6)  # Set to 8x6 inches

# 2. Now directly call scanpy's plotting function without any additional parameters
#    When it internally creates the figure, it will follow the global size set above
sc.pl.pca_variance_ratio(adata_myeloid_hvg, log=True, n_pcs=50)
plt.show()

# 3. Restore Matplotlib's default settings
#    This is a good practice to prevent this size setting from affecting your subsequent plots
plt.rcParams.update(plt.rcParamsDefault)

In [ ]:
print("\n--- Re-running Harmony on the myeloid subset ---")
sc.external.pp.harmony_integrate(
    adata_myeloid_hvg,
    key='orig.ident',
    basis='X_pca',
    adjusted_basis='X_pca_harmony',
    random_state=seed
)

# Transfer the results back to the main myeloid object
adata_myeloid.obsm['X_pca'] = adata_myeloid_hvg.obsm['X_pca']
adata_myeloid.obsm['X_pca_harmony'] = adata_myeloid_hvg.obsm['X_pca_harmony']



In [ ]:
del adata_myeloid_hvg
gc.collect()

In [ ]:
# --- Re-calculate neighbors, UMAP, and clustering ---
print("\n--- Re-calculating UMAP and sub-clustering ---")
sc.pp.neighbors(adata_myeloid, n_pcs=15, use_rep='X_pca_harmony', random_state=seed)
sc.tl.umap(adata_myeloid, random_state=seed)




In [ ]:
gene_list_main = [
    'CD3D', 'CD3E', 'CD3G',# T cells
    'NKG7',  'FGFBP2', # NK
    'KLRB1','TRAC','TRBC1','TRBC2', # NKT
    'MS4A1', 'CD79A', 'CD19', # B cell
    'IGHG1', 'MZB1',  # Plasma
    'LILRA4', 'IL3RA', 'CLEC4C', # pDC
    'CD8A', 'CD4', # T cell
    'TRGV9', 'TRDV2', # Gamma-Delta T cell
    #'PPBP','HBB'
    ]# Platelet and Erythrocyte


gene_list_myeloid = [
    'LYZ', 'S100A8', 'S100A9', 'VCAN', # Myeloid
    # cDC1 markers
    'XCR1',
    # cDC2 markers
    'CD1C','CCL22', 'CLEC10A','FCER1A',
    #Monocyte markers
    'CD14', 'FCGR3A','FCN1', 'S100A4', 'S100B','CD163',
    #Macrophage markers
    'C1QA', 'C1QC',
    #Microglia markers
     'P2RY12',
    #Mast cell markers
    'KIT','S100B',
    #ASDC
    'AXL','SIGLEC1'
]

In [185]:
mac_subset_marker_genes = [
    'APOE', 'C1QA', 'C1QC', 'CD3D', 'CD68', 'CXCL8', 'EPCAM', 'IL1B',
    'LYZ', 'MS4A1', 'PPBP', 'SQLE', 'SPP1', 'TREM2','C1QA','C1QC'
]
genes_to_plot = list(set(gene_list_myeloid + mac_subset_marker_genes))  

In [ ]:
sc.tl.leiden(adata_myeloid, resolution=4.0, key_added='leiden_myeloid', random_state=seed)
sc.pl.umap(adata_myeloid, color='leiden_myeloid', legend_loc='on data', title='Myeloid Sub-clusters')

sc.pl.umap(adata_myeloid, color=gene_list_main, legend_loc='on data')
sc.pl.umap(adata_myeloid, color=gene_list_myeloid, legend_loc='on data')



In [ ]:
# 在调用dotplot之前
print(f"Original number of genes to plot: {len(genes_to_plot)}")

# 找出实际存在于.raw中的基因
existing_genes = [gene for gene in genes_to_plot if gene in adata_myeloid.raw.var_names]

# 找出缺失的基因（用于调试和报告）
missing_genes = [gene for gene in genes_to_plot if gene not in adata_myeloid.raw.var_names]
if missing_genes:
    print(f"\nWarning: The following genes were not found in adata.raw.var_names and will be skipped:")
    print(missing_genes)

print(f"\nNumber of existing genes that will be plotted: {len(existing_genes)}")


genes_to_plot =existing_genes

In [ ]:
# --- Plot Dot Plot for Cell Type Annotation ---
# Dot plot is an excellent way to show the expression of specific genes in each cluster.
# The size of the dot represents the proportion of cells expressing the gene, and the color intensity represents the average expression level.
# **`use_raw=True`** is a key parameter that ensures we use the normalized and log-transformed raw expression values,
# rather than scaled values, which is more accurate for biological interpretation.
print("Generating dot plot...")
dotplot = sc.pl.dotplot(
    adata_myeloid,
    genes_to_plot,
    groupby="leiden_myeloid",
    use_raw=True,         # <-- Key! Use data from .raw
    #dendrogram=True,      # Perform hierarchical clustering on clusters, similar clusters will be grouped together
    standard_scale='var', # Normalize by gene to make genes with different expression levels comparable
    cmap='Reds',         # Use blue color scheme
    figsize=(18, 10),     # Set figure size
    return_fig=True,      # Return figure object for further modification
    show=False            # Do not display the figure immediately
)

# --- Customize and Beautify Dot Plot ---
print("Beautifying dot plot...")
dotplot.make_figure() # Generate the figure
main_ax = dotplot.ax_dict['mainplot_ax'] # Get the axes object of the main plot

# Add grid lines to make the figure easier to read
# Add horizontal lines (between genes)
main_ax.hlines(y=np.arange(len(genes_to_plot)) - 0.5, xmin=-0.5, xmax=len(adata_myeloid.obs['leiden_myeloid'].cat.categories) - 0.5, color='lightgray', lw=0.5, ls='--')
# Add vertical lines (between clusters)
main_ax.vlines(x=np.arange(len(adata_myeloid.obs['leiden_myeloid'].cat.categories)) - 0.5, ymin=-0.5, ymax=len(genes_to_plot) - 0.5, color='lightgray', lw=0.5, ls='--')

plt.show()


print("\n--- Exporting Raw Data of Dot Plot ---")
mean_expression_df = dotplot.dot_color_df
fraction_df = dotplot.dot_size_df

# with pd.option_context('display.max_rows', None, 'display.max_columns', None):
#     print("\n--- Mean Expression Matrix (Color) ---")
#     #print(mean_expression_df)
#     print("\n--- Cell Expression Fraction Matrix (Dot Size) ---")
#     #print(fraction_df)



# 2. Call the function to generate a summary table
final_summary_df = anno_self.generate_summary_table(mean_expression_df, fraction_df)

# 3. Convert the resulting DataFrame to a tab-separated string and print it
tsv_output = final_summary_df.to_csv(sep='\t')

#print("--- Gene Expression Level Summary Table (Tab-Separated) ---")
print(tsv_output)

In [204]:
import importlib
import anno_self
importlib.reload(anno_self)

In [205]:
# --- Perform Annotation Using apply Method (More Concise Than for Loop) ---
annotations_list = final_summary_df.apply(anno_self.annotate_csf_myeloid, axis=1)

# --- Create Final Concise Report ---
results_df = pd.DataFrame({
    'Cluster': final_summary_df.index,
    'Annotation': annotations_list
})


In [206]:

# --- Generate Explanation ---
results_df['Explanation'] = results_df.apply(
    anno_self.generate_myeloid_explanation, 
    axis=1, 
    original_df=final_summary_df
)

# --- Print Final Report ---
print("\n\n--- Final Annotation Report ---")
anno_self.pretty_print_df(results_df)


In [207]:
annotation_dict = results_df.set_index('Cluster')['Annotation'].to_dict()

In [219]:
annotation_dict

In [ ]:
#annotation_dict = {'0': 'Macrophage', '1': 'Classical Monocytes', '2': 'Macrophage', '3': 'Classical Monocytes', '4': 'Classical Monocytes', '5': 'Macrophage', '6': 'Macrophage', '7': 'Macrophage', '8': 'Macrophage', '9': 'Macrophage', '10': 'cDC2', '11': 'Intermediate Monocytes', '12': 'cDC2', '13': 'Macrophage', '14': 'Macrophage', '15': 'Macrophage', '16': 'Intermediate Monocytes', '17': 'cDC2', '18': 'Macrophage', '19': 'cDC2', '20': 'Macrophage', '21': 'cDC2', '22': 'cDC2', '23': 'Macrophage', '24': 'cDC2', '25': 'Macrophage', '26': 'cDC2', '27': 'cDC2', '28': 'Macrophage', '29': 'cDC2', '30': 'Macrophage', '31': 'cDC2', '32': 'Classical Monocytes', '33': 'Macrophage', '34': 'cDC2', '35': 'Macrophage', '36': 'Macrophage', '37': 'cDC2', '38': 'cDC1', '39': 'Macrophage', '40': 'Macrophage', '41': 'Macrophage', '42': 'Macrophage', '43': 'Macrophage', '44': 'Macrophage', '45': 'Macrophage', '46': 'Mast', '47': 'Macrophage', '48': 'Macrophage', '49': 'Macrophage', '50': 'cDC2', '51': 'Classical Monocytes'}


In [ ]:
# Apply mapping to create a new 'cell_subtype' column
adata_myeloid.obs['cell_subtype'] = adata_myeloid.obs['leiden_myeloid'].map(annotation_dict).astype('category')

# View the count of roughly annotated cells
print("--- Broad Annotation Counts ---")
print(adata_myeloid.obs['cell_subtype'].value_counts())

print("\n" + "="*30 + "\n")  # Separator



# Visualize the broad annotations
sc.pl.umap(adata_myeloid, color='cell_subtype',
           # legend_loc='on data',
           frameon=False,  # Remove the border
           title='Broad Cell Types')

In [ ]:
# 1. 定义所有巨噬细胞亚群（根据你的数据列出，排除非巨噬细胞类型）
macrophage_subtypes = [
    'Lipid-Associated Macrophage',  # LAM（实验组）
    'Macrophage-CD163',            # 其他巨噬细胞1
    'Macrophage-SPP1',             # 其他巨噬细胞2
    'Macrophage-IL1B'              # 其他巨噬细胞3
]

# 2. （可选，推荐）筛选仅包含巨噬细胞的adata子集（减少计算量，避免非巨噬细胞干扰）
adata_macrophage = adata_myeloid[adata_myeloid.obs['cell_subtype'].isin(macrophage_subtypes)].copy()

# 3. 构建分组：LAM  vs  其他巨噬细胞
adata_macrophage.obs['mac_group'] = 'Other_Macrophage'  # 默认设为“其他巨噬细胞”（对照组）
adata_macrophage.obs.loc[
    adata_macrophage.obs['cell_subtype'] == 'Lipid-Associated Macrophage', 
    'mac_group'
] = 'LAM'  # 实验组：LAM

# 查看分组数量，确认分组正确
print("分组细胞数量统计：")
print(adata_macrophage.obs['mac_group'].value_counts())

In [368]:
# 1. 提取top2000可变基因的名称（假设你的可变基因存储在adata_myeloid.var['highly_variable']中）
top2000_genes = adata_myeloid.var[adata_myeloid.var['highly_variable'] == True].index.tolist()

# 2. 筛选巨噬细胞子集（adata_macrophage）的基因列表为top2000可变基因
adata_macrophage_top2000 = adata_macrophage[:, top2000_genes].copy()

# 3. 基于top2000可变基因重新运行差异分析
sc.tl.rank_genes_groups(
    adata_macrophage_top2000,
    groupby='mac_group',
    groups=['LAM'],
    reference='Other_Macrophage',
    method='wilcoxon',
    logfc_threshold=0.25,
    adj_pval_threshold=0.05,
    min_group_fraction=0.1
)

# 4. 提取优化后的差异基因（方法同之前，过滤核糖体/管家基因）
de_top2000_df = sc.get.rank_genes_groups_df(adata_macrophage_top2000, group='LAM')
de_top2000_df_sig = de_top2000_df[de_top2000_df['pvals_adj'] < 0.05].copy()

# 过滤非特异性基因
exclude_prefixes = ['RPL', 'RPS', 'ATP']
de_top2000_df_filtered = de_top2000_df_sig[~de_top2000_df_sig['names'].str.startswith(tuple(exclude_prefixes))]

# 提取上调/下调Top10
up_top10_specific = de_top2000_df_filtered[de_top2000_df_filtered['scores'] > 0].sort_values('logfoldchanges', ascending=False).head(10)
down_top10_specific = de_top2000_df_filtered[de_top2000_df_filtered['scores'] < 0].sort_values('logfoldchanges', ascending=True).head(10)

# 打印结果
print("=== 基于top2000可变基因：LAM上调Top10（特征性）===")
print(up_top10_specific[['names', 'logfoldchanges', 'pvals_adj']])
print("\n=== 基于top2000可变基因：LAM下调Top10（特征性）===")
print(down_top10_specific[['names', 'logfoldchanges', 'pvals_adj']])

In [372]:
# 针对下调基因，过滤极端logfoldchanges（保留绝对值<5的）
de_top2000_df_filtered_down = de_top2000_df_filtered[
    (de_top2000_df_filtered['scores'] < 0) &  # 下调基因
    (abs(de_top2000_df_filtered['logfoldchanges']) < 5)  # 过滤极端值
]

# 提取下调Top10（按logfoldchanges升序）
down_top10_corrected = de_top2000_df_filtered_down.sort_values('logfoldchanges', ascending=True).head(10)

# 打印校正后的下调基因
print("=== 校正极端值后：LAM下调Top10（生物学合理）===")
print(down_top10_corrected[['names', 'logfoldchanges', 'pvals_adj']])

In [373]:
# 选2个下调基因（比如NR4A2、CCL4）
sc.pl.violin(
    adata_macrophage_top2000,
    keys=['AIF1', 'TUBA1B'],
    groupby='mac_group',
    rotation=45
)


In [ ]:
sc.pl.umap(adata_myeloid, color=mac_subset_marker_genes, legend_loc='on data')

In [376]:
adata.obs

In [378]:
# 查看一下 n_genes_by_counts,total_counts,total_counts_mt,pct_counts_mt各细胞类型的分布，绘制小提琴图
sc.pl.violin(
    adata_macrophage,
    keys=['n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt'],
    groupby='cell_subtype',
    rotation=45
)


In [ ]:
#保存adata_myeloid
adata_myeloid.write_h5ad('adata_myeloid.h5ad')

In [ ]:
# --- 步骤 1: 在主对象 adata 中创建或准备一个用于精细注释的总列 ---
# 我们统一使用 'cell_subtype' 作为最终的精细注释列名。
# 检查该列是否已存在。如果不存在，就用粗略注释 'cell_type' 来初始化它。。
if 'cell_subtype' not in adata.obs.columns:
    print("列 'cell_subtype' 不存在，正在基于 'cell_type' 创建...")
    adata.obs['cell_subtype'] = 'Unassigned'
    print("列 'cell_subtype' 已存在，将直接在其上更新 myeloid 细胞注释...")
    # 确保列是可修改的字符串类型
if pd.api.types.is_categorical_dtype(adata.obs['cell_subtype']):
    adata.obs['cell_subtype'] = adata.obs['cell_subtype'].astype(str)


# --- 步骤 2: 提取 myeloid 子集的精细注释 ---
print("\n--- 步骤 2: 提取 myeloid 子集的精细注释 ---")
myeloid_refined_annotations = adata_myeloid.obs['cell_subtype']


# --- 步骤 3: 使用 .loc 将 myeloid 的精细注释更新回主对象的新列中 ---
print("\n--- 步骤 3: 将 myeloid 注释合并回主对象的 'cell_subtype' 列 ---")
# 这个操作会根据细胞 barcode，精确地只更新那些属于 myeloid 细胞的行
adata.obs.loc[myeloid_refined_annotations.index, 'cell_subtype'] = myeloid_refined_annotations


# --- 步骤 4: 将新列转换为 'category' 类型以节省内存 ---
adata.obs['cell_subtype'] = adata.obs['cell_subtype'].astype('category')
print("\n合并完成。")

In [239]:
# 1. 正确定义mask：筛选cell_type为目标类别的细胞（用isin匹配多个值）
mask = adata.obs['cell_type'].isin(['B cell', 'Plasma cell', 'pDC'])

# 2. 找出需要添加的新类别
new_categories = pd.Index(adata.obs.loc[mask, 'cell_type'].unique()).difference(
    adata.obs['cell_subtype'].cat.categories
)

# 3. 添加新类别（如有）
if not new_categories.empty:
    print(f"正在为 'cell_subtype' 添加新的类别: {list(new_categories)}")
    adata.obs['cell_subtype'] = adata.obs['cell_subtype'].cat.add_categories(new_categories)

# 4. 赋值（仅执行一次）
adata.obs.loc[mask, 'cell_subtype'] = adata.obs.loc[mask, 'cell_type'].to_numpy()

print("操作成功！")

In [242]:
#将cell_type改为cell_type_broad
adata.obs['cell_type_broad'] = adata.obs['cell_type'].copy()
del adata.obs['cell_type']


In [245]:
gc.collect()

# --- Subset T-cells for re-analysis ---
adata_T = adata[adata.obs['cell_type_broad'] == 'T cell'].copy()
print(f"Successfully extracted {adata_T.n_obs} T-cells for sub-clustering.")

In [246]:
# --- Repeat the analysis pipeline for the T-cell subset ---

sc.pp.filter_genes(adata_T, min_cells=1)
print("\n--- Finding HVGs within the T-cell subset ---")
sc.pp.highly_variable_genes(adata_T, batch_key='orig.ident', n_top_genes=2000)

adata_T_hvg = adata_T[:, adata_T.var.highly_variable].copy()
sc.pp.scale(adata_T_hvg, max_value=10)
sc.tl.pca(adata_T_hvg, n_comps=30, random_state=seed)


In [248]:
# --- Drawing a Scree Plot ---
print("Drawing a scree plot to assist in determining the selection of n_pcs...")

# 1. Modify Matplotlib's global default figure size before plotting
#    This will affect all figures created afterwards
plt.rcParams['figure.figsize'] = (8, 6)  # Set to 8x6 inches

# 2. Now directly call scanpy's plotting function without any additional parameters
#    When it internally creates the figure, it will follow the global size set above
sc.pl.pca_variance_ratio(adata_T_hvg, log=True, n_pcs=50)
plt.show()

# 3. Restore Matplotlib's default settings
#    This is a good practice to prevent this size setting from affecting your subsequent plots
plt.rcParams.update(plt.rcParamsDefault)

In [249]:

print("\n--- Re-running Harmony on the T-cell subset ---")
sc.external.pp.harmony_integrate(
    adata_T_hvg,
    key='orig.ident',
    basis='X_pca',
    adjusted_basis='X_pca_harmony',
    random_state=seed
)

adata_T.obsm['X_pca'] = adata_T_hvg.obsm['X_pca']
adata_T.obsm['X_pca_harmony'] = adata_T_hvg.obsm['X_pca_harmony']

print("\n--- Re-calculating UMAP and sub-clustering ---")
sc.pp.neighbors(adata_T, n_pcs=15, use_rep='X_pca_harmony', random_state=seed)
sc.tl.umap(adata_T, random_state=seed)


In [250]:
sc.tl.leiden(adata_T, resolution=2.0, key_added='leiden_T', random_state=seed)

sc.pl.umap(adata_T, color='leiden_T', legend_loc='on data', title='T-cell Subclusters')

In [261]:
# 1. 定义你想要的完整基因列表
genes_requested = [
    'B3GAT1', 'CCR7', 'CD3D', 'CD3E', 'CD3G', 'CD4', 'CD68', 'CD69', 'CXCR4', 'LTB', 
    'RGS1', 'CD79A', 'CD8A', 'CD8B', 'CTLA4', 'CYTOR', 'EOMES', 'FOS', 'FOXP3', 'GATA3', 
    'GNLY', 'GZMA', 'GZMB', 'GZMK', 'HAVCR2', 'HLA-DRA', 'IGHG1', 'IKZF2', 'IL2RA', 
    'IL2RB', 'IL7R', 'ITGAE', 'JUN', 'KLRB1', 'KLRD1', 'KLRG1', 'LAG3', 'LYZ', 'MS4A1', 
    'MZB1', 'NKG7', 'PDCD1', 'PRF1', 'PTPRC', 'RORC', 'S100A4', 'SELL', 'SLC4A10', 
    'TBX21', 'TOP2A', 'TRDC', 'TRDV2', 'TRGV9', 'ISG15', 'IFIT1', 'IFI6', 'MX1', 'DDX60', 
    'IFNG', 'TCF7', 'TOX', 'BCL6', 'PRDM1', 'IRF4', 'CXCR3', 'CCR4', 'CCR6', 'CXCR5', 
    'CX3CR1', 'ENTPD1', 'SLAMF6', 'CD244', 'FAS', 'ICOS', 'CD27', 'CD28', 'TNFRSF9', 
    'IL2', 'TNF', 'IL4', 'IL5', 'IL13', 'IL17A', 'IL10', 'ZBTB16', 'MKI67'
]

# 2. 检查哪些基因在数据中真正可用
# 如果 use_raw=True，检查 adata_T.raw.var_names；否则检查 adata_T.var_names
if adata_T.raw is not None:
    available_genes = adata_T.raw.var_names
else:
    available_genes = adata_T.var_names

genes_to_plot = [gene for gene in genes_requested if gene in available_genes]

# 3. 打印出哪些基因被剔除了，方便你知晓数据情况
missing_genes = set(genes_requested) - set(genes_to_plot)
if missing_genes:
    print(f"以下基因在数据中未找到，已跳过: {missing_genes}")

# 4. 运行绘图
dotplot_T = sc.pl.dotplot(
    adata_T,
    genes_to_plot,
    groupby='leiden_T',
    use_raw=True,
    standard_scale='var',
    cmap='Reds',
    figsize=(16, 6), # 基因较多，建议调宽 figsize
    return_fig=True,
    show=True
)

In [262]:
sc.pl.dotplot(
    adata_T,
    genes_to_plot,
    groupby='leiden_T',
    use_raw=True,
    standard_scale='var',
    cmap='Reds',
    figsize=(12, 6),
    show=True
)

In [263]:
print("\n--- 正在导出点图的原始数据 ---")
mean_expression_df = dotplot_T.dot_color_df
fraction_df = dotplot_T.dot_size_df

final_summary_df = anno_self.generate_summary_table(mean_expression_df, fraction_df)

summaries = anno_self.generate_cluster_summaries(final_summary_df)

# 使用 option_context 临时修改设置，不影响后续代码
with pd.option_context('display.max_colwidth', None):
    print(summaries)


In [257]:
# 应用注释函数 (现在输入的数据形状是正确的了)
annotations_list = final_summary_df.apply(anno_self.annotate_t_cells_atlas_standard, axis=1)

# 创建报告DataFrame
results_df = pd.DataFrame({
    'Cluster': final_summary_df.index,
    'Annotation': annotations_list
})
#print(results_df)


In [259]:
# 生成解释
results_df['Explanation'] = results_df.apply(
    anno_self.generate_T_explanation_v11,
    axis=1,
    original_df=final_summary_df
)

# 打印最终报告
print("\n\n--- 最终注释报告 ---")
# 将簇的索引名设为'Cluster'，让输出更好看
results_df = results_df.reset_index(drop=True)
anno_self.pretty_print_df(results_df)

In [264]:
annotation_dict = results_df.set_index('Cluster')['Annotation'].to_dict()
print("\n--- 转换后的注释字典 ---")
#print(annotation_dict)

# 应用映射，创建新的 'cell_subtype' 列

adata_T.obs['cell_subtype'] = (
    adata_T.obs['leiden_T'].astype(str).map(annotation_dict).astype('category')
)


In [271]:
annotation_dict = {
    # --- CD4+ 系列 (谱系+状态，完全符合指南第14页要求) ---
    '0': 'CD4+ TN',
    '7': 'CD4+ TN',
    '19': 'CD4+ TN',
    '20': 'CD4+ TA',          # CD69++ 激活状态 (Table 1)
    '6': 'CD4+ TH1 M',        # 极化辅助状态 (Table 3)
    '9': 'CD4+ TH1 M',
    '25': 'CD4+ TH1 M',
    '2': 'CD4+ TH2 M',
    '8': 'CD4+ TH2 M',
    '12': 'CD4+ TH2 M',
    '11': 'CD4+ Treg M',      # 极化调节状态 (Table 3)
    '3': 'CD4+ TM',           # 数据显示 CD4(+)，无明显极化，规范标为 CD4+ TM
    '5': 'CD4+ TM',           # 数据显示 CD4(low)，归入 CD4+ 记忆群
    '13': 'CD4+ TM',          # 数据显示 CD4(+)，归入 CD4+ 记忆群
    '16': 'CD4+ TM',          # 数据显示 CD4(low)，归入 CD4+ 记忆群
    '21': 'CD4+ TM',          # 数据显示 CD4(low)，归入 CD4+ 记忆群
    '24': 'CD4+ T ISG',       # 谱系为 CD4 的干扰素状态
    
    # --- CD8+ 系列 (谱系+状态/模块代码) ---
    '14': 'CD8+ TN',
    '1': 'CD8+ TM',           # 传统的 CD8+ 记忆群
    '4': 'CD8+ TM',
    '18': 'CD8+ TM',
    '10': 'CD8+ TDM',         # 具有效应特征的播散性记忆 (依据 Fig 2 & Table 7)
    '15': 'CD8+ TDM',         # 具有高度细胞毒性的播散性记忆 (替代不规范的 Effector-like)
    '22': 'CD8+ TX',          # 耗竭状态 (Table 5)
    
    # --- 非常规/类固有亚群 (依据 Table 6) ---
    '23': 'MAIT',             # 专用术语，无需加 CD4/CD8
    '17': 'γδ+ TM'            # 专用术语
}
adata_T.obs['cell_subtype'] = (
    adata_T.obs['leiden_T'].astype(str).map(annotation_dict).astype('category')
)

In [272]:
from cProfile import label


sc.pl.umap(
    adata_T, 
    color=['cell_subtype'],
    cmap='Reds',          # 使用高对比度的红色系
    vmax='p99',           # 将颜色上限设为99%分位数，增强对比
    ncols=4              # 将3个基因图排成一行
)
sc.pl.umap(
    adata_T, 
    color=['leiden_T'],
    cmap='Reds',          # 使用高对比度的红色系
    vmax='p99',           # 将颜色上限设为99%分位数，增强对比
    ncols=4 ,
    legend_loc='on data'             # 将3个基因图排成一行
)

In [273]:
sc.pl.umap(
    adata_T, 
    color=['CD4','CD8A','CD8B'],
    ncols=3 ,
    legend_loc='on data'             # 将3个基因图排成一行
)

In [ ]:
#保存 adata_T
adata_T.write_h5ad('CSF_adata_T.h5ad',
    compression='gzip',
    compression_opts=5 )

In [ ]:
# --- 提取 T 子集的精细注释 ---
print("\n--- 提取 T 子集的精 fine 注释 ---")
# 从 adata_T 中提取 cell_subtype 列
T_refined_annotations = adata_T.obs['cell_subtype']


# --- 步骤 1 (关键修正): 在赋值前，确保目标列是可修改的类型 ---
# 检查 'cell_subtype' 列是否存在，如果不存在则创建
if 'cell_subtype' not in adata.obs.columns:
    print("列 'cell_subtype' 不存在，正在基于 'cell_type_broad' 创建...")
    adata.obs['cell_subtype'] = adata.obs['cell_type_broad'].astype(str)

# 如果 'cell_subtype' 列已经是 category 类型，先将其转换为字符串 (str) 类型
# 这是解决 TypeError 的核心步骤！
if pd.api.types.is_categorical_dtype(adata.obs['cell_subtype']):
    print("将 'cell_subtype' 列临时转换为字符串类型以便更新...")
    adata.obs['cell_subtype'] = adata.obs['cell_subtype'].astype(str)


# --- 步骤 2: 使用 .loc 将 T 细胞的精细注释更新回主对象 ---
print("\n--- 步骤 2: 将 T 细胞注释合并回主对象的 'cell_subtype' 列 ---")
# 现在，目标列是字符串类型，可以接受任何新的字符串值
adata.obs.loc[T_refined_annotations.index, 'cell_subtype'] = T_refined_annotations.astype(str)


# --- 步骤 3: (推荐) 操作完成后，将列转换回 'category' 类型以节省内存 ---
print("\n--- 步骤 3: 将更新后的列转换回 category 类型 ---")
adata.obs['cell_subtype'] = adata.obs['cell_subtype'].astype('category')
print("\n合并完成。")


# --- 步骤 4: 验证 ---
print("\n--- 验证主对象中的 'cell_subtype' 列 ---")
print("最终精细注释列的细胞类型计数:")
print(adata.obs['cell_subtype'].value_counts())

In [ ]:
gc.collect()

# --- Subset B cell for re-analysis ---
adata_B = adata[adata.obs['cell_type_broad'] == 'B cell lineage'].copy()
print(f"Successfully extracted {adata_B.n_obs} B cell for sub-clustering.")
adata_B.raw = adata_B.copy()
sc.pp.filter_genes(adata_B, min_cells=1)

sc.pp.highly_variable_genes(adata_B, batch_key='orig.ident', n_top_genes=2000)

adata_B_hvg = adata_B[:, adata_B.var.highly_variable].copy()
sc.pp.scale(adata_B_hvg, max_value=10)
sc.tl.pca(adata_B_hvg, n_comps=15, random_state=seed)

sc.external.pp.harmony_integrate(
    adata_B_hvg,
    key='orig.ident',
    basis='X_pca',
    adjusted_basis='X_pca_harmony',
    random_state=seed
)

adata_B.obsm['X_pca'] = adata_B_hvg.obsm['X_pca']
adata_B.obsm['X_pca_harmony'] = adata_B_hvg.obsm['X_pca_harmony']

sc.pp.neighbors(adata_B, n_pcs=15, use_rep='X_pca_harmony', random_state=seed)
sc.tl.umap(adata_B, random_state=seed)
sc.tl.leiden(adata_B, resolution=1.0, key_added='leiden_B', random_state=seed)


sc.pl.umap(adata_B, color=['IGHG1', 'MZB1','MS4A1','CD79A'],
    legend_loc='on data', title='NK Subclusters')

In [295]:
sc.pl.umap(adata_B, color='leiden_B',
    legend_loc='on data', title='B Subclusters')


In [296]:
B_subset_marker_genes =   ['IGHG1', 'MZB1','MS4A1','CD79A']  

In [298]:
import importlib
import anno_self
importlib.reload(anno_self)

dotplot_NK = sc.pl.dotplot(
   adata_B,
    B_subset_marker_genes,
    groupby='leiden_B',
    use_raw=True,
    standard_scale='var',
    cmap='Reds',
    figsize=(12, 6),
    return_fig=True,
    show=True
)


sc.pl.dotplot(
   adata_B,
    B_subset_marker_genes,
    groupby='leiden_B',
    use_raw=True,
    standard_scale='var',
    cmap='Reds',
    figsize=(12, 6),
    show=True
)

print("\n--- 正在导出点图的原始数据 ---")
mean_expression_df = dotplot_NK.dot_color_df
fraction_df = dotplot_NK.dot_size_df

final_summary_df = anno_self.generate_summary_table(mean_expression_df, fraction_df)
print("\n--- 基因表达水平总结表 ---")

summaries = anno_self.generate_cluster_summaries(final_summary_df)

# 使用 option_context 临时修改设置，不影响后续代码
with pd.option_context('display.max_colwidth', None):
    print(summaries)


In [300]:
cell_annotation = {
    0: "B cell",
    1: "B cell",
    2: "Plasma cell",
    3: "B cell",
    4: "B cell",
    5: "B cell",
    6: "Plasma cell",
    7: "Plasma cell",
    8: "Plasma cell",
    9: "B cell",
    10: "B cell",
    11: "B cell",
    12: "B cell"
}

In [303]:
# 重新运行映射
adata_B.obs['cell_subtype'] = adata_B.obs['leiden_B'].map(cell_annotation).astype('category')

# 检查一下是否有 NA
print(adata_B.obs['cell_subtype'].value_counts())

In [304]:
import matplotlib.pyplot as plt

print("--- Gene Expression on UMAP ---")
sc.pl.umap(
    adata_B, 
    color=['cell_subtype'],
    palette='Set2',       # 使用高对比度色板
    size=20,              # 调整点的大小
    frameon=False,        # 去掉外边框
    title='B Cell Subtypes Identification'
)
plt.show()

In [306]:
# 1. 打印当前所有的列名，确认是不是叫 'leiden_B'
print("当前obs中的列名:", adata_B.obs.columns.tolist())

# 2. 打印 'leiden_B' 的前几个值及其真实类型
sample_val = adata_B.obs['leiden_B'].iloc[0]
print(f"leiden_B 列的值示例: '{sample_val}'，类型: {type(sample_val)}")

# 3. 强制转换映射字典（同时支持数字和字符串，防止类型误差）
raw_map = {
    0: "B cell", 1: "B cell", 2: "Plasma cell", 3: "B cell", 
    4: "B cell", 5: "B cell", 6: "Plasma cell", 
    7: "Plasma cell", 8: "Plasma cell", 9: "B cell", 
    10: "B cell", 11: "B cell", 12: "B cell"
}
# 生成一个万无一失的字典：同时包含 "0" 和 0
final_map = {}
for k, v in raw_map.items():
    final_map[str(k)] = v
    final_map[int(k)] = v

# 4. 使用一种更强力的映射方法
# 先转为字符串，去掉可能存在的空格，再映射
adata_B.obs['cell_subtype'] = (
    adata_B.obs['leiden_B']
    .astype(str)           # 强转字符串
    .str.strip()           # 去掉空格
    .map(final_map)        # 映射
)

# 5. 检查映射结果（如果这里打印出来还是全为0或NaN，说明字典键值对还是没对上）
print("\n--- 映射后的计数检查 ---")
print(adata_B.obs['cell_subtype'].value_counts(dropna=False))

# 6. 绘图（增加 color_map 确保颜色出来）
import scanpy as sc
sc.pl.umap(adata_B, color='cell_subtype', frameon=False, size=30)

In [308]:
# --- 提取 NK 子集的精细注释 ---
print("\n--- 提取 NK 子集的精细注释 ---")
B_refined_annotations = adata_B.obs['cell_subtype']


# --- 步骤 1 (关键修正): 在赋值前，确保目标列是可修改的类型 ---
# 检查 'cell_subtype' 列是否存在，如果不存在则创建一个
if 'cell_subtype' not in adata.obs.columns:
    print("列 'cell_subtype' 不存在，正在基于 'cell_type_broad' 创建...")
    # 假设您有一个 'cell_type_broad' 列用于初始化
    adata.obs['cell_subtype'] = adata.obs['cell_type_broad'].astype(str)

# 如果 'cell_subtype' 列已经是 category 类型，先将其转换为字符串 (str) 类型
# 这是解决 TypeError 的核心步骤！
if pd.api.types.is_categorical_dtype(adata.obs['cell_subtype']):
    print("将 'cell_subtype' 列临时转换为字符串类型以便更新...")
    adata.obs['cell_subtype'] = adata.obs['cell_subtype'].astype(str)


# --- 步骤 2: 使用 .loc 将 NK 细胞的精细注释更新回主对象 ---
print("\n--- 步骤 2: 将 NK 注释合并回主对象的 'cell_subtype' 列 ---")
# 为了确保万无一失，将源数据也转换为字符串
adata.obs.loc[B_refined_annotations.index, 'cell_subtype'] = B_refined_annotations.astype(str)


# --- 步骤 3: (推荐) 操作完成后，将列转换回 'category' 类型以节省内存 ---
print("\n--- 步骤 3: 将更新后的列转换回 category 类型 ---")
adata.obs['cell_subtype'] = adata.obs['cell_subtype'].astype('category')
print("\n合并完成。")




In [311]:
adata.obs

In [313]:
#将cell_type_broad中的Ambiguous替换为Myeloid cell lineage
adata.obs['cell_type_broad'] = adata.obs['cell_type_broad'].replace('Ambiguous','Myeloid cell lineage')

In [ ]:
#将cell_type_broad列为NK cell lineage的所有cell的cell_type列设为NK cell lineage
# 1. 扩展cell_subtype列的分类类别，添加'NK cell'（若已存在则不报错）
adata.obs['cell_subtype'] = adata.obs['cell_subtype'].cat.add_categories(['NK cell'])

# 2. 执行你的赋值操作（此时'NK cell'已在类别列表中，可正常赋值）
adata.obs.loc[adata.obs['cell_type_broad'] == 'NK cell lineage', 'cell_subtype'] = 'NK cell'



In [ ]:
#将Lipid-Associated Macrophage改为Macrophage-CD163
# 方案1：基础替换（直接合并分类，推荐）
adata.obs['cell_subtype'] = adata.obs['cell_subtype'].replace(
    to_replace='Lipid-Associated Macrophage',  # 待合并的旧分类
    value='Macrophage-CD163'                   # 目标已有分类
)

# 关键补充：清理冗余分类（移除已无对应细胞的Lipid-Associated Macrophage标签）
# 执行后，分类列表中仅保留有细胞的唯一分类，更简洁
adata.obs['cell_subtype'] = adata.obs['cell_subtype'].cat.remove_unused_categories()

# 验证结果：查看合并后的分类分布
print("合并后细胞亚群分布：")
print(adata.obs['cell_subtype'].value_counts())


In [380]:
adata.obs['cell_subtype'].value_counts()

In [387]:
#将Lipid-Associated Macrophage改为Macrophage-CD163
# 方案1：基础替换（直接合并分类，推荐）
adata.obs['cell_type_broad'] = adata.obs['cell_type_broad'].replace(
    to_replace='T cell',  # 待合并的旧分类
    value='T cell lineage'                   # 目标已有分类
)

# 关键补充：清理冗余分类（移除已无对应细胞的Lipid-Associated Macrophage标签）
# 执行后，分类列表中仅保留有细胞的唯一分类，更简洁
adata.obs['cell_type_broad'] = adata.obs['cell_type_broad'].cat.remove_unused_categories()

# 验证结果：查看合并后的分类分布
print("合并后细胞亚群分布：")
print(adata.obs['cell_type_broad'].value_counts())


In [358]:
#删除nan值
adata.obs['cell_subtype'] = adata.obs['cell_subtype'].cat.remove_unused_categories()

In [388]:
sc.pl.umap(adata,color = ['cell_subtype','cell_type_broad'],size=5, ncols = 1)

In [402]:
print(adata.obs['cell_subtype'].unique().to_list())

In [417]:
# --- 运行逻辑 ---
optimized_subtype_order = [
    'Macrophage-IL1B', 'Macrophage-SPP1', 'Macrophage-CD163','MAIT', 'γδ+ TM', 
    'Classical Monocytes', 'Intermediate Monocytes', 
    'CD4+ TN', 'CD4+ TM', 'CD4+ TH1 M', 'CD4+ TH2 M', 'CD4+ Treg M', 'CD4+ T ISG', 'CD4+ TA',
    'CD8+ TN', 'CD8+ TM', 'CD8+ TDM', 'CD8+ TX','NK cell',    
    'cDC1', 'cDC2', 'pDC', 'Mast',
    'B cell', 'Plasma cell'
]


In [418]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec

# 强制白色风格
plt.style.use('default') 
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'sans-serif']
plt.rcParams['axes.facecolor'] = 'white'

def plot_compact_composition_v8(adata, sample_col='sample_ID', cluster_col='cell_subtype', 
                                group_col='disease', palette=None, subtype_order=None):
    # 1. 数据准备 (保持原逻辑)
    meta = adata.obs[[sample_col, group_col, cluster_col]].copy()
    meta[sample_col] = meta[sample_col].astype(str)
    meta[group_col] = meta[group_col].astype(str)
    
    sample_totals = meta.groupby(sample_col, observed=True).size()
    counts = meta.groupby([sample_col, group_col, cluster_col], observed=True).size().unstack(fill_value=0)
    props = counts.div(counts.sum(axis=1), axis=0) * 100
    props.index = props.index.get_level_values(sample_col)
    
    if subtype_order is not None:
        existing_order = [x for x in subtype_order if x in props.columns]
        others = [x for x in props.columns if x not in existing_order]
        plot_order = existing_order + others
    else:
        plot_order = props.mean().sort_values(ascending=False).index.tolist()
    
    props = props[plot_order] 
    sample_info = meta[[sample_col, group_col]].drop_duplicates().set_index(sample_col).sort_values(group_col)
    props = props.loc[sample_info.index]
    sample_totals = sample_totals.loc[sample_info.index]
    
    # 坐标计算
    unique_groups = sample_info[group_col].unique()
    group_gap = 3.0 
    x_coords, current_x, group_boundaries = [], 0, []
    for group in unique_groups:
        group_samples = sample_info[sample_info[group_col] == group].index
        start_x = current_x
        for _ in group_samples:
            x_coords.append(current_x); current_x += 1
        group_boundaries.append((start_x, current_x - 1, group))
        current_x += group_gap

    # 3. 配色
    if palette is None:
        import seaborn as sns
        palette = sns.color_palette("tab20", n_colors=len(props.columns))
    color_map = dict(zip(props.columns, palette))

    # 4. 绘图布局 - 关键优化：大幅减小 hspace
    fig = plt.figure(figsize=(24, 10), dpi=100, facecolor='white')
    # 【修复1】：hspace 从 0.12 减小到 0.02，让子图几乎贴在一起
    gs = GridSpec(3, 1, height_ratios=[1.2, 7, 0.4], hspace=0.02)
    
    ax_top = fig.add_subplot(gs[0])
    ax_main = fig.add_subplot(gs[1])
    ax_bot = fig.add_subplot(gs[2])

    for ax in [ax_top, ax_main, ax_bot]:
        ax.set_facecolor('white')

    # --- A. 顶部总量图 ---
    ax_top.bar(x_coords, sample_totals.values, color='#BDC3C7', width=0.8, zorder=3)
    ax_top.set_ylabel('Total Cells', fontweight='bold', fontsize=13)
    ax_top.spines[['top', 'right', 'bottom', 'left']].set_visible(False)
    ax_top.set_xticks([])
    ax_top.yaxis.grid(True, linestyle='--', alpha=0.15, color='gray')

    # --- B. 中部占比堆叠图 ---
    bottom = np.zeros(len(x_coords))
    for cluster in props.columns:
        ax_main.bar(x_coords, props[cluster].values, bottom=bottom, label=cluster, 
                    color=color_map[cluster], width=0.88, edgecolor='white', linewidth=0.1, zorder=3)
        bottom += props[cluster].values

    ax_main.set_ylabel('Cell Percentage (%)', fontweight='bold', fontsize=13)
    ax_main.set_ylim(0, 100)
    ax_main.spines[['top', 'right', 'bottom', 'left']].set_visible(False)
    ax_main.set_xticks([]) 
    ax_main.yaxis.grid(True, linestyle='--', alpha=0.15, color='gray')

    # --- C. 底部疾病分组Bar + 样本名 ---
    # 【修复2】：调整 ax_bot 的内部坐标，使色块贴近上方，样本名贴近色块
    group_colors = plt.cm.Pastel1(np.linspace(0, 1, len(unique_groups)))
    for i, (start, end, name) in enumerate(group_boundaries):
        # 让色块占据 ax_bot 的顶部 (0.7 到 1.0)
        rect = patches.FancyBboxPatch((start - 0.45, 0.7), (end - start + 0.9), 0.25, 
                                     boxstyle="round,pad=0.01", facecolor=group_colors[i], 
                                     edgecolor='none', clip_on=False)
        ax_bot.add_patch(rect)
        # 疾病名文字居中
        ax_bot.text((start + end) / 2, 0.82, name, ha='center', va='center', 
                    fontweight='bold', fontsize=14, color='#34495E')

    ax_bot.set_xlim(ax_top.get_xlim())
    ax_bot.set_ylim(0, 1) # 限制 y 轴范围，使 xticks 靠上
    ax_bot.set_xticks(x_coords)
    ax_bot.set_xticklabels(sample_info.index, rotation=45, ha='right', 
                            rotation_mode='anchor', fontsize=9, color='#2C3E50')
    
    ax_bot.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
    ax_bot.set_yticks([])
    # 【修复3】：pad 从 12 减小到 2，让文字直接紧贴色块下方
    ax_bot.tick_params(axis='x', which='both', length=0, pad=2) 

    # --- D. 图例 ---
    handles, labels = ax_main.get_legend_handles_labels()
    ax_main.legend(handles[::-1], labels[::-1], title=r'$\mathbf{Cell\ Subtypes}$', 
                   bbox_to_anchor=(1.01, 0.5), loc='center left', frameon=False, fontsize=10)

    # 调整整体边距，确保不被切掉
    plt.subplots_adjust(right=0.88, left=0.08, bottom=0.15, top=0.95)
    return fig

# 调用
fig = plot_compact_composition_v8(
    adata, 
    subtype_order=optimized_subtype_order[::-1]
)
plt.show()

In [419]:
def merge_cell_subtypes(x):
    x = str(x)
    if x.startswith('CD4+'):
        return 'CD4+ T'
    elif x.startswith('CD8+'):
        return 'CD8+ T'
    elif x in ['B cell', 'Plasma cell']:
        return 'B cell lineage'
    else:
        return x

# 在 adata.obs 中新增列
adata.obs['cell_subtype1'] = adata.obs['cell_subtype'].apply(merge_cell_subtypes)

# 打印查看合并后的结果
print(adata.obs['cell_subtype1'].unique())

In [432]:
# 更新后的排序列表
new_subtype_order = [
    'Macrophage-IL1B', 'Macrophage-SPP1', 'Macrophage-CD163','Classical Monocytes', 'Intermediate Monocytes', 
    'cDC1', 'cDC2', 'pDC', 'Mast',
    'NK cell',
    'CD4+ T', 
    'CD8+ T',
    'MAIT', 'γδ+ TM',
    
    'B cell lineage'
]

In [436]:
# 你提供的优化顺序
new_subtype_order = [
    'Macrophage-IL1B', 
    'Macrophage-SPP1', 
    'Macrophage-CD163',
    'Classical Monocytes', 
    'Intermediate Monocytes', 
    'cDC1', 
    'cDC2', 
    'pDC', 
    'Mast',
    'NK cell',
    'CD4+ T', 
    'CD8+ T',
    'MAIT', 
    'γδ+ TM',
    'B cell lineage'
]

# 调用绘图函数
# 注意：不需要再对列表加 [[::-1]]，函数内部处理逻辑会让列表第一个元素在最下方
fig = plot_final_composition_with_age_text(
    adata, 
    sample_col='sample_ID', 
    cluster_col='cell_subtype1', 
    age_col='age', 
    subtype_order=new_subtype_order[::-1]
)

plt.show()

In [389]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap
from scipy.cluster.hierarchy import linkage, dendrogram

def plot_final_perfect_roe_heatmap(adata, cell_type_col='cell_subtype', broad_type_col='cell_type_broad', 
                                   sample_col='sample_ID', group_col='disease'):
    # --- 1. 数据准备 ---
    obs_df = adata.obs[[sample_col, group_col, cell_type_col, broad_type_col]].copy()
    for col in obs_df.columns:
        obs_df[col] = obs_df[col].astype(str)
    obs_df = obs_df[obs_df[cell_type_col] != 'nan']
    
    # 计算 ROE
    cross_tab = pd.crosstab(obs_df[cell_type_col], obs_df[sample_col])
    row_sums = cross_tab.sum(axis=1).values.reshape(-1, 1)
    col_sums = cross_tab.sum(axis=0).values.reshape(1, -1)
    total_sum = cross_tab.values.sum()
    expected = (row_sums @ col_sums) / total_sum
    roe_df = np.log2((cross_tab / expected) + 0.1)

    # --- 2. 疾病组聚类 ---
    sample_to_group = obs_df[[sample_col, group_col]].drop_duplicates().set_index(sample_col)
    group_means = roe_df.T.groupby(sample_to_group[group_col]).mean()
    group_linkage = linkage(group_means, method='ward')
    group_dendro = dendrogram(group_linkage, no_plot=True)
    clustered_group_order = group_means.index[group_dendro['leaves']].tolist()
    
    # 排序
    sample_meta = sample_to_group.copy()
    sample_meta[group_col] = pd.Categorical(sample_meta[group_col], categories=clustered_group_order, ordered=True)
    sample_meta = sample_meta.sort_values(group_col)
    unique_groups = clustered_group_order

    # --- 3. 亚群排序与行高计算 ---
    broad_order = ['T cell lineage', 'NK cell lineage','B cell lineage', 'Myeloid cell lineage', 'pDC']
    broad_types = [b for b in broad_order if b in obs_df[broad_type_col].unique()]
    
    height_ratios = []
    broad_to_subtypes = {}
    total_rows = 0
    for bt in broad_types:
        subtypes = obs_df[obs_df[broad_type_col] == bt][cell_type_col].unique()
        sub_data = roe_df.loc[subtypes, sample_meta.index]
        if len(sub_data) > 1:
            Z = linkage(sub_data, 'ward')
            reorder_idx = dendrogram(Z, no_plot=True)['leaves']
            sorted_sub = sub_data.index[reorder_idx[::-1]].tolist()
        else:
            sorted_sub = sub_data.index.tolist()
        broad_to_subtypes[bt] = sorted_sub
        height_ratios.append(len(sorted_sub))
        total_rows += len(sorted_sub)

    # --- 4. 【核心】正方形格子计算 ---
    # 设定每个格子的物理尺寸 (inches)
    cell_size = 0.35 
    n_cols = len(sample_meta)
    
    heatmap_width = n_cols * cell_size
    heatmap_height = total_rows * cell_size
    
    # 边距预留
    left_side = 4.0   # 大类名
    tree_side = 1.0   # 左侧树
    right_side = 4.0  # 右侧标签
    bottom_side = 5.0 # 底部标签 + 聚类树
    top_side = 0.5
    
    fig_width = heatmap_width + left_side + tree_side + right_side
    fig_height = heatmap_height + bottom_side + top_side
    
    fig = plt.figure(figsize=(fig_width, fig_height))

    # GridSpec 布局：热图块 -> 疾病Bar -> 聚类线
    # 这里的 height_ratios 末尾增加了两行：Bar 和 Dendrogram
    main_gs = GridSpec(len(broad_types) + 2, 4, 
                       height_ratios=height_ratios + [1.2, 2.5], 
                       width_ratios=[0.12, 0.05, 0.63, 0.20], 
                       hspace=0.2, wspace=0.01)

    vmax = 10 
    colors = ["#053061", "#4393c3", "#ffffff", "#d6604d", "#67001f"]
    my_cmap = LinearSegmentedColormap.from_list("custom_RdBu", list(zip([0.0, 0.25, 0.5, 0.75, 1.0], colors)))

    # --- 5. 绘制热图块 ---
    for i, bt in enumerate(broad_types):
        sub_data = roe_df.loc[broad_to_subtypes[bt], sample_meta.index]
        n_sub = len(sub_data)
        
        # A. 大类名
        ax_text = fig.add_subplot(main_gs[i, 0])
        ax_text.text(0.95, 0.5, bt, ha='right', va='center', fontweight='bold', fontsize=16)
        ax_text.axis('off')

        # B. 左侧树
        ax_tree = fig.add_subplot(main_gs[i, 1])
        if n_sub > 1:
            Z = linkage(sub_data, 'ward')
            d = dendrogram(Z, orientation='left', ax=ax_tree, color_threshold=0, above_threshold_color='black', no_labels=True)
            max_d = np.array(d['dcoord']).max()
            ax_tree.set_xlim(max_d * 1.05, 0)
            ax_tree.set_ylim(0, 10 * n_sub)
        ax_tree.axis('off')

        # C. 热图 (square=False，通过figsize保证)
        ax_heatmap = fig.add_subplot(main_gs[i, 2])
        sns.heatmap(sub_data, ax=ax_heatmap, cmap=my_cmap, center=0, vmin=-vmax, vmax=vmax, cbar=False, 
                    xticklabels=False, yticklabels=True, square=False)
        ax_heatmap.set_ylim(n_sub, 0)
        ax_heatmap.set_ylabel(""); ax_heatmap.set_xlabel("")

        # 白色分割线
        curr_pos = 0
        for group in unique_groups[:-1]:
            curr_pos += sum(sample_meta[group_col] == group)
            ax_heatmap.axvline(curr_pos, color='white', linewidth=2)
            
        ax_heatmap.yaxis.tick_right()
        ax_heatmap.set_yticklabels(sub_data.index, rotation=0, fontsize=13)
        ax_heatmap.tick_params(axis='y', length=0)

    # --- 6. 绘制疾病Bar与名称 ---
    ax_bar = fig.add_subplot(main_gs[len(broad_types), 2])
    all_groups_orig = adata.obs[group_col].unique()
    group_palette = sns.color_palette("Set2", len(all_groups_orig))
    color_dict = dict(zip(all_groups_orig, group_palette))
    
    group_widths = [sum(sample_meta[group_col] == g) for g in unique_groups]
    group_centers = []
    x_pos = 0
    for group, width in zip(unique_groups, group_widths):
        ax_bar.barh(0, width, left=x_pos, color=color_dict[group], height=0.6)
        # 疾病名标签
        ax_bar.text(x_pos + width/2, -1.0, group, ha='right', va='top', rotation=45, fontweight='bold', fontsize=14)
        group_centers.append(x_pos + width/2)
        x_pos += width
        
    ax_bar.set_xlim(0, n_cols)
    ax_bar.set_ylim(-2.0, 1.0)
    ax_bar.axis('off')

    # --- 7. 【修复核心】绘制最底部的疾病聚类线 ---
    ax_bottom_tree = fig.add_subplot(main_gs[len(broad_types) + 1, 2])
    
    icoord = np.array(group_dendro['icoord'])
    dcoord = np.array(group_dendro['dcoord'])
    
    # 映射逻辑：将默认叶子坐标 (5, 15, 25...) 对应到实际的 group_centers
    default_leaf_pos = [j * 10 + 5 for j in range(len(unique_groups))]
    
    def map_group_coords(x):
        # 使用锚点插值，而非全量线性插值
        return np.interp(x, default_leaf_pos, group_centers)

    max_h = dcoord.max()
    for xs, ys in zip(icoord, dcoord):
        # 绘制倒挂的树：枝尖(ys=0)在最上方(靠近文字)，根部在下方
        ax_bottom_tree.plot(map_group_coords(xs), max_h - ys, color='black', lw=1.5)
    
    ax_bottom_tree.set_xlim(0, n_cols)
    # 给树留出一点垂直空间
    ax_bottom_tree.set_ylim(-max_h*0.1, max_h * 1.1)
    ax_bottom_tree.axis('off')

    # --- 8. 颜色条 ---
    cbar_ax = fig.add_axes([0.96, 0.45, 0.008, 0.12]) 
    sm = plt.cm.ScalarMappable(cmap=my_cmap, norm=plt.Normalize(-vmax, vmax))
    cb = plt.colorbar(sm, cax=cbar_ax, ticks=[-10, -5, 0, 5, 10])
    cb.ax.set_title('Exp', fontweight='bold', fontsize=14, pad=15)
    cb.outline.set_visible(False)

    plt.show()

# 调用
plot_final_perfect_roe_heatmap(adata)

In [438]:
#提取CD4+ TH1 M 和 CD4+ TH2 M 细胞
adata_TH = adata[adata.obs['cell_subtype'].isin(['CD4+ TH1 M', 'CD4+ TH2 M'])]

In [439]:
sc.pl.umap(adata_TH, color=['CD4', 'CD8A','CD8B'])


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap
from scipy.cluster.hierarchy import linkage, dendrogram

def plot_final_perfect_roe_heatmap(adata, cell_type_col='cell_subtype', broad_type_col='cell_type_broad', 
                                   sample_col='sample_ID', group_col='disease'):
    # --- 1. 数据准备 ---
    obs_df = adata.obs[[sample_col, group_col, cell_type_col, broad_type_col]].copy()
    for col in obs_df.columns:
        obs_df[col] = obs_df[col].astype(str)
    obs_df = obs_df[obs_df[cell_type_col] != 'nan']
    
    # 计算 ROE
    cross_tab = pd.crosstab(obs_df[cell_type_col], obs_df[sample_col])
    row_sums = cross_tab.sum(axis=1).values.reshape(-1, 1)
    col_sums = cross_tab.sum(axis=0).values.reshape(1, -1)
    total_sum = cross_tab.values.sum()
    expected = (row_sums @ col_sums) / total_sum
    roe_df = np.log2((cross_tab / expected) + 0.1)

    # --- 2. 疾病组聚类 ---
    sample_to_group = obs_df[[sample_col, group_col]].drop_duplicates().set_index(sample_col)
    group_means = roe_df.T.groupby(sample_to_group[group_col]).mean()
    group_linkage = linkage(group_means, method='ward')
    group_dendro = dendrogram(group_linkage, no_plot=True)
    clustered_group_order = group_means.index[group_dendro['leaves']].tolist()
    
    # 排序
    sample_meta = sample_to_group.copy()
    sample_meta[group_col] = pd.Categorical(sample_meta[group_col], categories=clustered_group_order, ordered=True)
    sample_meta = sample_meta.sort_values(group_col)
    unique_groups = clustered_group_order

    # --- 3. 亚群排序与行高计算 ---
    broad_order = ['T cell lineage', 'NK cell lineage','B cell lineage', 'Myeloid cell lineage', 'pDC']
    broad_types = [b for b in broad_order if b in obs_df[broad_type_col].unique()]
    
    height_ratios = []
    broad_to_subtypes = {}
    total_rows = 0
    for bt in broad_types:
        subtypes = obs_df[obs_df[broad_type_col] == bt][cell_type_col].unique()
        sub_data = roe_df.loc[subtypes, sample_meta.index]
        if len(sub_data) > 1:
            Z = linkage(sub_data, 'ward')
            reorder_idx = dendrogram(Z, no_plot=True)['leaves']
            sorted_sub = sub_data.index[reorder_idx[::-1]].tolist()
        else:
            sorted_sub = sub_data.index.tolist()
        broad_to_subtypes[bt] = sorted_sub
        height_ratios.append(len(sorted_sub))
        total_rows += len(sorted_sub)

    # --- 4. 【核心】正方形格子计算 ---
    # 设定每个格子的物理尺寸 (inches)
    cell_size = 0.35 
    n_cols = len(sample_meta)
    
    heatmap_width = n_cols * cell_size
    heatmap_height = total_rows * cell_size
    
    # 边距预留
    left_side = 4.0   # 大类名
    tree_side = 1.0   # 左侧树
    right_side = 4.0  # 右侧标签
    bottom_side = 5.0 # 底部标签 + 聚类树
    top_side = 0.5
    
    fig_width = heatmap_width + left_side + tree_side + right_side
    fig_height = heatmap_height + bottom_side + top_side
    
    fig = plt.figure(figsize=(fig_width, fig_height))

    # GridSpec 布局：热图块 -> 疾病Bar -> 聚类线
    # 这里的 height_ratios 末尾增加了两行：Bar 和 Dendrogram
    main_gs = GridSpec(len(broad_types) + 2, 4, 
                       height_ratios=height_ratios + [1.2, 2.5], 
                       width_ratios=[0.12, 0.05, 0.63, 0.20], 
                       hspace=0.2, wspace=0.01)

    vmax = 10 
    colors = ["#053061", "#4393c3", "#ffffff", "#d6604d", "#67001f"]
    my_cmap = LinearSegmentedColormap.from_list("custom_RdBu", list(zip([0.0, 0.25, 0.5, 0.75, 1.0], colors)))

    # --- 5. 绘制热图块 ---
    for i, bt in enumerate(broad_types):
        sub_data = roe_df.loc[broad_to_subtypes[bt], sample_meta.index]
        n_sub = len(sub_data)
        
        # A. 大类名
        ax_text = fig.add_subplot(main_gs[i, 0])
        ax_text.text(0.95, 0.5, bt, ha='right', va='center', fontweight='bold', fontsize=16)
        ax_text.axis('off')

        # B. 左侧树
        ax_tree = fig.add_subplot(main_gs[i, 1])
        if n_sub > 1:
            Z = linkage(sub_data, 'ward')
            d = dendrogram(Z, orientation='left', ax=ax_tree, color_threshold=0, above_threshold_color='black', no_labels=True)
            max_d = np.array(d['dcoord']).max()
            ax_tree.set_xlim(max_d * 1.05, 0)
            ax_tree.set_ylim(0, 10 * n_sub)
        ax_tree.axis('off')

        # C. 热图 (square=False，通过figsize保证)
        ax_heatmap = fig.add_subplot(main_gs[i, 2])
        sns.heatmap(sub_data, ax=ax_heatmap, cmap=my_cmap, center=0, vmin=-vmax, vmax=vmax, cbar=False, 
                    xticklabels=False, yticklabels=True, square=False)
        ax_heatmap.set_ylim(n_sub, 0)
        ax_heatmap.set_ylabel(""); ax_heatmap.set_xlabel("")

        # 白色分割线
        curr_pos = 0
        for group in unique_groups[:-1]:
            curr_pos += sum(sample_meta[group_col] == group)
            ax_heatmap.axvline(curr_pos, color='white', linewidth=2)
            
        ax_heatmap.yaxis.tick_right()
        ax_heatmap.set_yticklabels(sub_data.index, rotation=0, fontsize=13)
        ax_heatmap.tick_params(axis='y', length=0)

    # --- 6. 绘制疾病Bar与名称 ---
    ax_bar = fig.add_subplot(main_gs[len(broad_types), 2])
    all_groups_orig = adata.obs[group_col].unique()
    group_palette = sns.color_palette("Set2", len(all_groups_orig))
    color_dict = dict(zip(all_groups_orig, group_palette))
    
    group_widths = [sum(sample_meta[group_col] == g) for g in unique_groups]
    group_centers = []
    x_pos = 0
    for group, width in zip(unique_groups, group_widths):
        ax_bar.barh(0, width, left=x_pos, color=color_dict[group], height=0.6)
        # 疾病名标签
        ax_bar.text(x_pos + width/2, -1.0, group, ha='right', va='top', rotation=45, fontweight='bold', fontsize=14)
        group_centers.append(x_pos + width/2)
        x_pos += width
        
    ax_bar.set_xlim(0, n_cols)
    ax_bar.set_ylim(-2.0, 1.0)
    ax_bar.axis('off')

    # --- 7. 【修复核心】绘制最底部的疾病聚类线 ---
    ax_bottom_tree = fig.add_subplot(main_gs[len(broad_types) + 1, 2])
    
    icoord = np.array(group_dendro['icoord'])
    dcoord = np.array(group_dendro['dcoord'])
    
    # 映射逻辑：将默认叶子坐标 (5, 15, 25...) 对应到实际的 group_centers
    default_leaf_pos = [j * 10 + 5 for j in range(len(unique_groups))]
    
    def map_group_coords(x):
        # 使用锚点插值，而非全量线性插值
        return np.interp(x, default_leaf_pos, group_centers)

    max_h = dcoord.max()
    for xs, ys in zip(icoord, dcoord):
        # 绘制倒挂的树：枝尖(ys=0)在最上方(靠近文字)，根部在下方
        ax_bottom_tree.plot(map_group_coords(xs), max_h - ys, color='black', lw=1.5)
    
    ax_bottom_tree.set_xlim(0, n_cols)
    # 给树留出一点垂直空间
    ax_bottom_tree.set_ylim(-max_h*0.1, max_h * 1.1)
    ax_bottom_tree.axis('off')

    # --- 8. 颜色条 ---
    cbar_ax = fig.add_axes([0.96, 0.45, 0.008, 0.12]) 
    sm = plt.cm.ScalarMappable(cmap=my_cmap, norm=plt.Normalize(-vmax, vmax))
    cb = plt.colorbar(sm, cax=cbar_ax, ticks=[-10, -5, 0, 5, 10])
    cb.ax.set_title('Exp', fontweight='bold', fontsize=14, pad=15)
    cb.outline.set_visible(False)

    plt.show()

# 调用
plot_final_perfect_roe_heatmap(adata)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap
from scipy.cluster.hierarchy import linkage, dendrogram

def plot_final_perfect_roe_heatmap(adata, cell_type_col='cell_subtype', broad_type_col='cell_type', 
                                   sample_col='orig.ident', group_col='disease', age_col='age'):
    # --- 1. 数据准备 ---
    # 确保提取年龄列
    obs_df = adata.obs[[sample_col, group_col, cell_type_col, broad_type_col, age_col]].copy()
    for col in [sample_col, group_col, cell_type_col, broad_type_col]:
        obs_df[col] = obs_df[col].astype(str)
    
    # 转换年龄为数值（如果不是的话）
    obs_df[age_col] = pd.to_numeric(obs_df[age_col], errors='coerce')
    obs_df = obs_df[obs_df[cell_type_col] != 'nan']
    
    # 计算 ROE
    cross_tab = pd.crosstab(obs_df[cell_type_col], obs_df[sample_col])
    row_sums = cross_tab.sum(axis=1).values.reshape(-1, 1)
    col_sums = cross_tab.sum(axis=0).values.reshape(1, -1)
    total_sum = cross_tab.values.sum()
    expected = (row_sums @ col_sums) / total_sum
    roe_df = np.log2((cross_tab / expected) + 0.1)

    # --- 2. 疾病组聚类 ---
    # 获取每个样本对应的疾病和年龄（取均值或唯一值，假设每个样本对应一个年龄）
    sample_meta = obs_df[[sample_col, group_col, age_col]].drop_duplicates().set_index(sample_col)
    
    group_means = roe_df.T.groupby(sample_meta[group_col]).mean()
    group_linkage = linkage(group_means, method='ward')
    group_dendro = dendrogram(group_linkage, no_plot=True)
    clustered_group_order = group_means.index[group_dendro['leaves']].tolist()
    
    # 样本排序
    sample_meta[group_col] = pd.Categorical(sample_meta[group_col], categories=clustered_group_order, ordered=True)
    sample_meta = sample_meta.sort_values(group_col)
    unique_groups = clustered_group_order
    n_cols = len(sample_meta)

    # --- 3. 亚群排序与行高计算 ---
    broad_order = ['T cell lineage', 'NK cell lineage','B cell lineage', 'Myeloid cell lineage', 'pDC']
    broad_types = [b for b in broad_order if b in obs_df[broad_type_col].unique()]
    
    height_ratios = []
    broad_to_subtypes = {}
    total_rows = 0
    for bt in broad_types:
        subtypes = obs_df[obs_df[broad_type_col] == bt][cell_type_col].unique()
        sub_data = roe_df.loc[subtypes, sample_meta.index]
        if len(sub_data) > 1:
            Z = linkage(sub_data, 'ward')
            reorder_idx = dendrogram(Z, no_plot=True)['leaves']
            sorted_sub = sub_data.index[reorder_idx[::-1]].tolist()
        else:
            sorted_sub = sub_data.index.tolist()
        broad_to_subtypes[bt] = sorted_sub
        height_ratios.append(len(sorted_sub))
        total_rows += len(sorted_sub)

    # --- 4. 布局计算 ---
    cell_size = 0.35 
    heatmap_width = n_cols * cell_size
    heatmap_height = total_rows * cell_size
    
    left_side, tree_side, right_side = 4.0, 1.0, 4.0
    bottom_side, top_side = 5.5, 0.5 # 稍微增加底部空间
    
    fig_width = heatmap_width + left_side + tree_side + right_side
    fig_height = heatmap_height + bottom_side + top_side
    
    fig = plt.figure(figsize=(fig_width, fig_height))

    # GridSpec 布局：热图块 -> Age Bar -> Disease Bar -> Dendrogram
    # 增加一行用于 Age Bar [0.4], 调整原先的 [1.2, 2.5]
    main_gs = GridSpec(len(broad_types) + 3, 4, 
                       height_ratios=height_ratios + [0.4, 0.6, 2.5], 
                       width_ratios=[0.12, 0.05, 0.63, 0.20], 
                       hspace=0.15, wspace=0.01)

    vmax = 10 
    colors = ["#053061", "#4393c3", "#ffffff", "#d6604d", "#67001f"]
    my_cmap = LinearSegmentedColormap.from_list("custom_RdBu", list(zip([0.0, 0.25, 0.5, 0.75, 1.0], colors)))

    # --- 5. 绘制热图块 ---
    for i, bt in enumerate(broad_types):
        sub_data = roe_df.loc[broad_to_subtypes[bt], sample_meta.index]
        n_sub = len(sub_data)
        
        ax_text = fig.add_subplot(main_gs[i, 0])
        ax_text.text(0.95, 0.5, bt, ha='right', va='center', fontweight='bold', fontsize=16)
        ax_text.axis('off')

        ax_tree = fig.add_subplot(main_gs[i, 1])
        if n_sub > 1:
            Z = linkage(sub_data, 'ward')
            d = dendrogram(Z, orientation='left', ax=ax_tree, color_threshold=0, above_threshold_color='black', no_labels=True)
            max_d = np.array(d['dcoord']).max()
            ax_tree.set_xlim(max_d * 1.05, 0)
            ax_tree.set_ylim(0, 10 * n_sub)
        ax_tree.axis('off')

        ax_heatmap = fig.add_subplot(main_gs[i, 2])
        sns.heatmap(sub_data, ax=ax_heatmap, cmap=my_cmap, center=0, vmin=-vmax, vmax=vmax, cbar=False, 
                    xticklabels=False, yticklabels=True, square=False)
        ax_heatmap.set_ylim(n_sub, 0)
        
        # 白色垂直分割线（区分疾病组）
        curr_pos = 0
        for group in unique_groups[:-1]:
            curr_pos += sum(sample_meta[group_col] == group)
            ax_heatmap.axvline(curr_pos, color='white', linewidth=2)
            
        ax_heatmap.yaxis.tick_right()
        ax_heatmap.set_yticklabels(sub_data.index, rotation=0, fontsize=13)
        ax_heatmap.set_ylabel(""); ax_heatmap.set_xlabel("")
        ax_heatmap.tick_params(axis='y', length=0)

    # --- 6. 绘制 Age Bar (新加) ---
    ax_age = fig.add_subplot(main_gs[len(broad_types), 2])
    # 提取排序后的样本年龄
    age_values = sample_meta[age_col].values.reshape(1, -1)
    # 使用 'YlGn' 或 'viridis' 渐变色表示年龄
    sns.heatmap(age_values, ax=ax_age, cmap="YlGnBu", cbar=False, 
                xticklabels=False, yticklabels=False)
    # 添加 "Age" 文字标签
    ax_age.set_ylabel("Age", rotation=0, ha='right', va='center', fontweight='bold', fontsize=12)
    # 同样添加组分割线
    curr_pos = 0
    for group in unique_groups[:-1]:
        curr_pos += sum(sample_meta[group_col] == group)
        ax_age.axvline(curr_pos, color='white', linewidth=2)

    # --- 7. 绘制疾病 Bar ---
    ax_bar = fig.add_subplot(main_gs[len(broad_types) + 1, 2])
    all_groups_orig = adata.obs[group_col].unique()
    group_palette = sns.color_palette("Set2", len(all_groups_orig))
    color_dict = dict(zip(all_groups_orig, group_palette))
    
    group_widths = [sum(sample_meta[group_col] == g) for g in unique_groups]
    group_centers = []
    x_pos = 0
    for group, width in zip(unique_groups, group_widths):
        ax_bar.barh(0, width, left=x_pos, color=color_dict[group], height=0.8)
        # 疾病名标签
        ax_bar.text(x_pos + width/2, -0.8, group, ha='right', va='top', rotation=45, fontweight='bold', fontsize=14)
        group_centers.append(x_pos + width/2)
        x_pos += width
        
    ax_bar.set_xlim(0, n_cols)
    ax_bar.set_ylim(-1.5, 1.0)
    ax_bar.axis('off')

    # --- 8. 绘制底部的疾病聚类树 ---
    ax_bottom_tree = fig.add_subplot(main_gs[len(broad_types) + 2, 2])
    icoord = np.array(group_dendro['icoord'])
    dcoord = np.array(group_dendro['dcoord'])
    default_leaf_pos = [j * 10 + 5 for j in range(len(unique_groups))]
    
    def map_group_coords(x):
        return np.interp(x, default_leaf_pos, group_centers)

    max_h = dcoord.max()
    for xs, ys in zip(icoord, dcoord):
        ax_bottom_tree.plot(map_group_coords(xs), max_h - ys, color='black', lw=1.5)
    
    ax_bottom_tree.set_xlim(0, n_cols)
    ax_bottom_tree.set_ylim(-max_h*0.1, max_h * 1.1)
    ax_bottom_tree.axis('off')

    # --- 9. 颜色条 (ROE 和 Age) ---
    # ROE Colorbar
    cbar_ax = fig.add_axes([0.96, 0.55, 0.008, 0.12]) 
    sm = plt.cm.ScalarMappable(cmap=my_cmap, norm=plt.Normalize(-vmax, vmax))
    cb = plt.colorbar(sm, cax=cbar_ax, ticks=[-10, -5, 0, 5, 10])
    cb.ax.set_title('Log2ROE', fontweight='bold', fontsize=12, pad=15)
    cb.outline.set_visible(False)

    # Age Colorbar
    age_cbar_ax = fig.add_axes([0.96, 0.35, 0.008, 0.10])
    age_min, age_max = obs_df[age_col].min(), obs_df[age_col].max()
    sm_age = plt.cm.ScalarMappable(cmap="YlGnBu", norm=plt.Normalize(age_min, age_max))
    cb_age = plt.colorbar(sm_age, cax=age_cbar_ax)
    cb_age.ax.set_title('Age', fontweight='bold', fontsize=12, pad=10)
    cb_age.outline.set_visible(False)

    plt.show()

# 调用
plot_final_perfect_roe_heatmap(adata, age_col='age_column_name')